In [6]:
#### [AI-CONTEXT]
#### ID: 001
#### ROLE: Import data straight from a Google Sheet that gathers closing quotations from the stock exchange of Brazil.
#### INPUT: User starting the application.
#### OUTPUT: Import the file StockIndex.csv from the local disk and load it to the code platform as a dataframe.

#### Dataset format: Each row is a stock, and the quantity of rows may vary.
#### Dataset format: The first column 'COMPANY_TICKER' holds the companies' tickers.
#### Dataset format: The second column 'COMPANY' holds the companies' names.
#### Dataset format: The third column 'CODE' classifies how the stocks are listed, traded, and governed on the Brazilian exchange.
#### Dataset format: The fourth column 'INDEX' carries the stocks' indexes.
#### Dataset format: The fifth column 'ETFs' indicates which ETFs the stocks belong to.
#### Dataset format: The sixth column 'DESCRIPTION' has a description of the company.
#### Dataset format: The seventh column 'SECTOR' indicates to which economic sector the company belongs.
#### Dataset format: The eighth column 'SUBSECTOR' indicates to which economic subsector the company belongs.
#### Dataset format: The ninth column 'B3_TRADING_SEGMENT' indicates the corporate governance of each stock.
#### Dataset format: The tenth column 'LIQUIDITY' indicates the percentage of 61 days that the stock was traded.
#### Dataset format: The last 61 columns are the last 61 workday market prices. These columns were named from left to right as -61, -60, -59, and so on until 0. The values in column 0 are the most recent (today's values). The values in the column -61 are the oldest ones.


import os
import pandas as pd
import urllib.request

SHEET_ID = "116eRXm29DXPY4f6aCJgQc9BM-1LQDfL2xMd1Ay8IHlg"  
SHEET_NAME = "StockIndex"  
LOCAL_PATH = '/home/andre/AI_Lab/20260623_Project_StockMarket_App/StockIndex.csv'

url = f"https://google.com{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}"
df = None

# Step 1: Try downloading live data with a proper timeout
try:
    print("🔄 Attempting to fetch live data from Google Sheets...")
    # Using urllib to handle the timeout natively before passing to pandas
    with urllib.request.urlopen(url, timeout=5) as response:
        df = pd.read_csv(response)
    print("✅ Live data loaded successfully!")
except Exception as e:
    print(f"⚠️  Network/DNS error detected ({e}). Switching to local fallback...")
    
    # Step 2: Fallback to local file if download fails
    if os.path.exists(LOCAL_PATH):
        df = pd.read_csv(LOCAL_PATH)
        print(f"📦 Successfully loaded local file: {LOCAL_PATH}")
    else:
        print(f"❌ Critical Error: Local fallback file not found at {LOCAL_PATH}")

# Step 3: Run diagnostics only if data was loaded from either source
if df is not None:
    print("=" * 70)
    print("🔍 [ID: 001 ROUTINE CHECK - CELL RUNNING]")
    print("=" * 70)
    print(f"• Loaded DataFrame Row Count        : {df.shape[0]}")
    print(f"• Loaded DataFrame Column Count     : {df.shape[1]}")
    print(f"• Name of First Column in File      : {df.columns[0]}")
    print(f"• Name of Penultimate Column in File: {df.columns[-2]}")
    print(f"• Name of Last Column in File       : {df.columns[-1]}")
    print("=" * 70)

🔄 Attempting to fetch live data from Google Sheets...
⚠️  Network/DNS error detected (<urlopen error [Errno -2] Name or service not known>). Switching to local fallback...
📦 Successfully loaded local file: /home/andre/AI_Lab/20260623_Project_StockMarket_App/StockIndex.csv
🔍 [ID: 001 ROUTINE CHECK - CELL RUNNING]
• Loaded DataFrame Row Count        : 1933
• Loaded DataFrame Column Count     : 71
• Name of First Column in File      : COMPANY_TICKER
• Name of Penultimate Column in File: -1
• Name of Last Column in File       : 0


In [7]:
    #### [AI-CONTEXT]
    #### ID: 002
    #### ROLE: '#N/A' cleanse.
    #### INPUT: Complete data import in the previous cell (successful execution of ID: 001).
    #### OUTPUT: Coerce string data conversion into float numbers, and then replace nan values with the closest float value on#### [AI-CONTEXT]
    
    
    import pandas as pd
    import numpy as np
    
    # Track the last 61 columns explicitly
    target_cols = df.columns[-61:]
    last_col_name = df.columns[-1]
    
    # Step 1: Clean raw string characters across target columns
    df[target_cols] = df[target_cols].replace(['N/A', '#N/A', '', r'^\s*$'], np.nan, regex=True)
    
    # Step 2: Replace comma decimal separators with periods before numeric coercion
    for col in target_cols:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    
    # Step 3: FORCE a hard conversion of the data type to float
    type_mapping = {col: 'float64' for col in target_cols}
    df = df.astype(type_mapping)
    
    # Step 4: Drop rows where ALL values in the price columns are NaN
    # Moved upstream to eliminate empty shell rows before processing sequence steps
    df = df.dropna(subset=target_cols, how='all')
    
    # Step 5: Chronologically correct fill - Forward fill (axis=1) first
    # This ensures missing days are filled using the closest PAST price, preventing future leakage!
    df[target_cols] = df[target_cols].ffill(axis=1)
    
    # Step 6: Backward fill fallback ONLY for leading edge boundary cells (e.g., Column -61)
    # If the very first historical day was NaN, this safely grabs the next closest future close
    df[target_cols] = df[target_cols].bfill(axis=1)
    
    # Step 7: Ensure the last column (Column 0 / Today) is cleanly rounded to 2 decimal places
    df[last_col_name] = df[last_col_name].round(2)
    
    # =====================================================================
    # DIAGNOSTIC ENGINE LAYER: CHECK FOR PRE-STANDARDIZATION DATAFRAME SIZE
    # =====================================================================
    
    print("=" * 70)
    print("🔍 [ID: 002 ROUTINE CHECK - CELL RUNNING]")
    print("=" * 70)
    print(f"• Valid rows remaining after drop  : {len(df)}")
    print(f"• Total NaN values remaining       : {df[target_cols].isna().sum().sum()}")
    print("=" * 70)

🔍 [ID: 002 ROUTINE CHECK - CELL RUNNING]
• Valid rows remaining after drop  : 1765
• Total NaN values remaining       : 0


In [ ]:
#### [AI-CONTEXT]
#### ID: 003
#### ROLE: Implement data cleanup to improve memory efficiency.
#### INPUT: Completing the 'N/A' cleanse in the previous cell (successful execution of ID: 002).
#### OUTPUT: Free up space used to allocate the huge_df dataframe.


# Immediately free up RAM space by explicitly deleting the object and triggering the Garbage Collector.

import gc

# 1. Delete the reference to the old dataframe
del target_cols, last_col_name

# 2. Force Python to collect the 'garbage' and release the RAM
gc.collect()

In [ ]:
#### [AI-CONTEXT]
#### ID: 004
#### ROLE: reorder Columns.
#### INPUT: Complete memory wiping in the previous cell (successful execution of ID: 003).
#### OUTPUT: Reorder the columns of the "df" dataframe.


import numpy as np

# 1. Capture the exact text labels for your 61 trailing price columns chronologically
# In the original 71-column layout, prices run from index 10 to the end
price_cols = list(df.columns[10:])

# 2. Combine your desired metadata order with the trailing price columns list
# Note: This list explicitly drops 'COMPANY_TICKER' and 'ETFs'
ordered_columns = [
    'INDEX', 'COMPANY', 'CODE', 'DESCRIPTION', 
    'SECTOR', 'SUBSECTOR', 'B3_TRADING_SEGMENT', 'LIQUIDITY'
] + price_cols

# 3. Safely reorder and filter using the layout list
df = df[ordered_columns].copy()

# Show the new layout structure
print("📋 Modified Dataframe Layout (Ticker Suppressed):")
print("-" * 65)
print(f"Total Columns Remaining: {df.shape[1]}")
print(f"Index 0 Identifier Name: {df.columns[0]}")
print(f"Index 3 NLP Feature Name: {df.columns[3]}")
print("-" * 65)

In [ ]:
#### [AI-CONTEXT]
#### ID: 005
#### ROLE: Preparation for the tokenization process.
#### INPUT: Reorder the columns of the "df" dataframe (successful execution of ID: 004).
#### OUTPUT: Load the HuggingFace Access Token in the notebook, by retrieving the HF_TOKEN API key indicated in the file named ".env".


import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami

# 1. Load the HF_TOKEN from your .env file
load_dotenv(dotenv_path="/home/andre/jupyter_notebook/20260623_Project_StockMarket_App/.env")

# 2. Get the token from environment variables
api_key = os.getenv("HF_TOKEN")

# 3. Log in and Verify
if api_key:
    try:
        # Perform programmatic login
        login(token=api_key)
        
        # Verify the identity of the logged-in user
        user_info = whoami()
        username = user_info.get("Andre Soares", "andre-alss")
        
        print(f"✅ Successfully logged into Hugging Face as: {username}\n")
    except Exception as e:
        print(f"❌ Login failed: {str(e)}\n")
else:
    print("⚠️ Warning: HF_TOKEN not found in .env file.\n")

In [ ]:
#### [AI-CONTEXT]
#### ID: 006
#### ROLE: Tokenization process for the NPL Branch.
#### INPUT: Successfully load the HuggingFace Access Token in the previous cell (successful execution of ID: 005).
#### OUTPUT: Encode the strings related to significant fund details in preparation for an NLP Branch of a future Multilayer Perceptron (MLP).


from transformers import AutoTokenizer

# 1. Initialize the specialized financial tokenizer
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# 2. Combine all significant fund details into a unified semantic context string per asset
# This preserves inter-variable relationships and guarantees a single, uniform tensor shape
unified_text_series = (
    "Index: " + df['INDEX'].astype(str).fillna("") + " | " +
    "Segment: " + df['B3_TRADING_SEGMENT'].astype(str).fillna("") + " | " +
    "Code: " + df['CODE'].astype(str).fillna("") + " | " +
    "Sector: " + df['SECTOR'].astype(str).fillna("") + " | " +
    "Subsector: " + df['SUBSECTOR'].astype(str).fillna("") + " | " +
    "Description: " + df['DESCRIPTION'].astype(str).fillna("")
).tolist()

# 3. Encode the entire text dataset in a single highly optimized pass
# max_length=512 ensures the long description fits comfortably within FinBERT's absolute limit
nlp_input_tensors = tokenizer(
    unified_text_series, 
    padding=True, 
    truncation=True, 
    max_length=512,
    return_tensors="pt"
)

# Show final tensor layout ready for the MLP network embedding layer
print("🚀 Tokenization complete!")
print(f"Input IDs shape:  {nlp_input_tensors['input_ids'].shape}")
print(f"Attention Mask shape: {nlp_input_tensors['attention_mask'].shape}")

In [ ]:
#### [AI-CONTEXT]
#### ID: 007
#### ROLE: Vectorization process for the NPL Branch.
#### INPUT: Successfully encode the fund details in the previous cell (successful execution of ID: 006).
#### OUTPUT: The Detail tensor that will feed a NLP Branch of the future Multilayer Perceptron (MLP).


import torch

# 1. Directly extract the complete, combined input token IDs from the unified dictionary
# 'nlp_input_tensors' is the object generated and confirmed in your previous execution step
details_input_tensor = nlp_input_tensors['input_ids']

# 2. Extract the exact sequence dimension length of the tensor layout
details_length = details_input_tensor.shape[1]

# Show the validated structural shape matching your target model entry point
print("🚀 NLP Branch Entry Tensor Validated:")
print("-" * 50)
print(f"Final Tensor Shape:  {details_input_tensor.shape}")
print(f"Tokens Per Asset (Length): {details_length}")
print("-" * 50)

In [ ]:
#### [AI-CONTEXT]
#### ID: 008
#### ROLE: Implement data cleanup to make the script memory efficient.
#### INPUT: Complete vectorization process for the NPL Branch in the previous cell (successful execution of ID: 007).
#### OUTPUT: Free up space used to allocate the input strings.


# Immediately free up RAM space by explicitly deleting the object and triggering the Garbage Collector.

import gc

del unified_text_series, nlp_input_tensors
gc.collect()

In [ ]:
#### [AI-CONTEXT]
#### ID: 009
#### ROLE: Vectorization and 80/20 train/eval split for Tabular Branch, NPL Branch, and Time-Series LSTM Branch.
#### INPUT: Complete memory wiping in the previous cell (successful execution of ID: 008).
#### OUTPUT: Tensors that will feed the branches of the future Multilayer Perceptron (MLP) in both train and eval sets.


import torch
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Locates the exact column label positions to ensure text or empty fields never enter the matrix
start_col_idx = df.columns.get_loc("-60")
end_col_idx = df.columns.get_loc("0") + 1

# Slice the matrix based on true column positions rather than relative negative indexing
numpy_matrix = df.iloc[:, start_col_idx:end_col_idx].values.astype(np.float64)


# 2. Isolate the liquidity column and convert to a structured 2D NumPy array
liquidity_df = df[['LIQUIDITY']].copy()
numpy_liquidity = liquidity_df.to_numpy().astype(np.float32).reshape(-1, 1)

# 3. FIXED: Added 'numpy_liquidity' to the input parameters to match the 6 outputs
text_train_np, text_eval_np, price_train_np, price_eval_np, liquidity_train_np, liquidity_eval_np = train_test_split(
    details_input_tensor.numpy(), 
    numpy_matrix, 
    numpy_liquidity,
    train_size=0.8, 
    test_size=0.2, 
    shuffle=False, # NEVER SHUFFLE
    random_state=68
)

# 4. Convert all partitions directly to PyTorch tensors
price_train_tensor = torch.tensor(price_train_np, dtype=torch.float32)
price_eval_tensor = torch.tensor(price_eval_np, dtype=torch.float32)

liquidity_train_tensor = torch.tensor(liquidity_train_np, dtype=torch.float32)
liquidity_eval_tensor = torch.tensor(liquidity_eval_np, dtype=torch.float32)

details_train_tensor = torch.tensor(text_train_np, dtype=torch.long)
details_eval_tensor = torch.tensor(text_eval_np, dtype=torch.long)

print(f"📊 Balanced Data Partitions:")
print("-" * 55)
print(f"Train Prices: {price_train_tensor.shape} | Train Text: {details_train_tensor.shape} | Train Liquidity: {liquidity_train_tensor.shape} ")
print(f"Eval Prices:  {price_eval_tensor.shape}  | Eval Text:  {details_eval_tensor.shape}  | Eval Liquidity:  {liquidity_eval_tensor.shape}")
print("-" * 55)

In [ ]:
#### [AI-CONTEXT]
#### ID: 010
#### ROLE: Diagnostic and statistical evaluation.
#### INPUT: Generate the numpy_matrix in the previous cell (successful execution of ID: 009).
#### OUTPUT: Plot a boxplot next to a normal distribution plot to check the discrepancy of the outliers before standardization.


import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

# 1. Extract raw data from the last 61 columns and flatten it
# .astype(float) and .flatten() handle any lingering structure issues
raw_flattened = df.iloc[:, -61:].values.astype(float).flatten()

# 2. Filter out any remaining NaNs so they don't break the histogram or statistics calculations
raw_flattened = raw_flattened[~np.isnan(raw_flattened)]

# 3. Set up a 1x2 grid layout figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Boxplot ---
ax1.boxplot(raw_flattened, vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightcoral', color='darkred'),
            medianprops=dict(color='black', linewidth=2))
ax1.set_title('Boxplot of Raw (Unstandardized) Data')
ax1.set_xlabel('Raw Units')
ax1.grid(True, linestyle='--', alpha=0.6)

# --- Plot 2: Histogram + Normal Curve Fit ---
count, bins, ignored = ax2.hist(raw_flattened, bins=50, density=True, 
                                alpha=0.6, color='salmon', edgecolor='black')

# Calculate the actual mean and standard deviation of the raw dataset
mu_raw = np.mean(raw_flattened)
sigma_raw = np.std(raw_flattened)

# Generate points along the X-axis mapping the specific spread of raw data
x_axis = np.linspace(mu_raw - 4 * sigma_raw, mu_raw + 4 * sigma_raw, 200)
gaussian_curve = stats.norm.pdf(x_axis, mu_raw, sigma_raw)

# Render the theoretical normal distribution curve (safely escaping backslashes)
ax2.plot(x_axis, gaussian_curve, color='darkred', linewidth=2.5, 
         label=f'Normal Fit\n($\\mu$={mu_raw:.2f}, $\\sigma$={sigma_raw:.2f})')

ax2.set_title('Raw Distribution vs. Normal Curve Fit')
ax2.set_xlabel('Raw Units')
ax2.set_ylabel('Probability Density')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

# Adjust spacing and display the figure
plt.tight_layout()
plt.show()

In [ ]:
#### [AI-CONTEXT]
#### ID: 011
#### ROLE: Standardization for the Time-Series LSTM Branch.
#### INPUT: Generate the price_tensor (successful execution of ID: 009).
#### OUTPUT: Scales the historical stock prices with a single, row-wise local/stock Standard Scaler.


import torch
import numpy as np

# FIX: Map directly to your clean sequential training partition from ID 009
if 'price_train_tensor' not in globals():
    raise NameError("❌ Sequence Error: 'price_train_tensor' partition not found. Please run ID:009 first.")

# =====================================================================
# 1. ENFORCE ABSOLUTE PRICE ARCHITECTURE (TRAIN SET)
# =====================================================================
p_train_f32 = price_train_tensor.to(torch.float32)

# =====================================================================
# 2. COMPUTE SCALING FACTOR EXCLUDING TARGET WINDOW (LEAK-PROOF)
# =====================================================================
# FIXED: Calculate mean and standard deviation strictly over the first 60 days
# This isolates column 0 (the target) to completely eliminate lookahead bias!
row_means = p_train_f32[:, :-1].mean(dim=1, keepdim=True)
row_stds = p_train_f32[:, :-1].std(dim=1, unbiased=False, keepdim=True) 
eps = 1e-6

# =====================================================================
# 3. EXECUTE ROW-WISE LOCAL STANDARD NORMALIZATION
# =====================================================================
# Broadcast the leak-proof parameters across the full 61-day timeline matrix safely
standardized_tensor = (p_train_f32 - row_means) / (row_stds + eps)

# =====================================================================
# VERIFICATION AND VOLATILITY PROFILE DIAGNOSTIC
# =====================================================================
print("📊 DIAGNÓSTICO DE PRE-PROCESSAMENTO (Row-Wise Local Price Mode - TRAIN ONLY):")
print("-" * 75)
print(f"   • Formato do Tensor Padronizado: {list(standardized_tensor.shape)} [Price Waves]")
print("-" * 75)
print(f"   • Formato das Matrizes de Média/Desvio: {list(row_means.shape)}")
print("-" * 75)
print(f"   • Ação 0 Média (Lookback-Only) : {standardized_tensor[0, :-1].mean().item():.4f} (Deve ser ~0)")
print(f"   • Ação 0 Desvio (Lookback-Only): {standardized_tensor[0, :-1].std(unbiased=False).item():.4f} (Deve ser ~1)")
print("-" * 75)

In [ ]:
#### [AI-CONTEXT]
#### ID: 012
#### ROLE: Statistical evaluation and diagnostic. Apply a hard statistical threshold to pull the extreme tails back into a healthy distribution.
#### INPUT: Generate the standardized_tensor in the previous cell (successful execution of ID: 011).
#### OUTPUT: Plot a boxplot next to a normal distribution plot to check the discrepancy of the outliers.


import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import torch

# =====================================================================
# 1. OUTLIER MITIGATION LAYER (SYMMETRICAL CLIPPING MATRIX)
# =====================================================================
# Hard clamp boundaries at +/- 3.5 standard deviations protect the LSTM
clipping_limit = 3.5

# Apply the clamp directly to the active tensor
standardized_tensor = torch.clamp(standardized_tensor, min=-clipping_limit, max=clipping_limit)

# =====================================================================
# 2. CONVERT PYTORCH TENSOR TO FLATTENED NUMPY FOR DIAGNOSTICS
# =====================================================================
flattened_data = standardized_tensor.detach().cpu().numpy().flatten()

# =====================================================================
# 3. SET UP VISUAL DIAGNOSTIC PLOT ENVIRONMENT
# =====================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Boxplot (Now bounded cleanly within +/- 3.5) ---
ax1.boxplot(flattened_data, vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='blue'),
            medianprops=dict(color='red', linewidth=2))
ax1.set_title('Boxplot of Outlier-Clipped Standardized Data', fontsize=12, fontweight='bold')
ax1.set_xlabel('Standardized Units (z-score)')
ax1.grid(True, linestyle='--', alpha=0.6)

# --- Plot 2: Histogram + Normal Curve Fit ---
count, bins, ignored = ax2.hist(flattened_data, bins=50, density=True, 
                                alpha=0.6, color='skyblue', edgecolor='black')

# Recalculate Gaussian tracking metrics based on the newly managed distribution
mu = np.mean(flattened_data)
sigma = np.std(flattened_data)
x_axis = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
gaussian_curve = stats.norm.pdf(x_axis, mu, sigma)

# Plot theoretical curve with corrected LaTeX escape backslashes for Greek variables
ax2.plot(x_axis, gaussian_curve, color='darkblue', linewidth=2.5, 
         label=f'Normal Fit\n($\\mu$={mu:.2f}, $\\sigma$={sigma:.2f})')

ax2.set_title('Clipped Return Distribution vs. Normal Curve', fontsize=12, fontweight='bold')
ax2.set_xlabel('Standardized Units (z-score)')
ax2.set_ylabel('Probability Density')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

# Adjust spacing and display the figure
plt.tight_layout()
plt.show()

In [ ]:
#### [AI-CONTEXT]
#### ID: 013
#### ROLE: Vectorization and standardization for the Time-Series LSTM Branch.
#### INPUT: Complete vectorization and 80/20 train/eval split (successful execution of ID: 009).
#### OUTPUT: price_std_train_tensor, price_std_eval_tensor, and price_std_target_train_tensor, all normalized with global Z-score standardization.


import torch
import numpy as np

if 'df' in globals():
    # =====================================================================
    # 1. ENFORCE HIGH-PERFORMANCE ABSOLUTE PRICE TIMELINE
    # =====================================================================
    p_train_f32 = price_train_tensor.to(torch.float32)
    p_eval_f32 = price_eval_tensor.to(torch.float32)
    
    # 2. SEPARATE FEATURES (DAYS 1-60) AND TARGETS (DAY 61) BEFORE SCALING
    x_train_raw = p_train_f32[:, :-1]   # First 60 absolute price steps (Features)
    y_train_raw = p_train_f32[:, [-1]]  # The 61st absolute price step (Target)
    
    x_eval_raw = p_eval_f32[:, :-1]     # First 60 absolute price steps (Features)
    y_eval_raw = p_eval_f32[:, [-1]]    # The 61st absolute price step (Target)

    # =====================================================================
    # 3. COMPUTE ROW-WISE LOCAL PARAMETERS (FROM TRAINING FEATURES ONLY)
    # =====================================================================
    # Calculates a distinct mean and std for each individual stock sequence row
    train_row_means = x_train_raw.mean(dim=1, keepdim=True)
    train_row_stds = x_train_raw.std(dim=1, unbiased=False, keepdim=True)
    
    eval_row_means = x_eval_raw.mean(dim=1, keepdim=True)
    eval_row_stds = x_eval_raw.std(dim=1, unbiased=False, keepdim=True)
    eps = 1e-6

    # =====================================================================
    # 4. EXECUTE ROW-WISE LOCAL Z-SCORE STANDARDIZATION
    # =====================================================================
    x_train_std = (x_train_raw - train_row_means) / (train_row_stds + eps)
    x_eval_std = (x_eval_raw - eval_row_means) / (eval_row_stds + eps)
    
    # Scale targets using their corresponding historical row parameters to prevent leakage
    y_train_std = (y_train_raw - train_row_means) / (train_row_stds + eps)
    y_eval_std = (y_eval_raw - eval_row_means) / (eval_row_stds + eps)

    # =====================================================================
    # 5. MITIGATION LAYER: SYMMETRICAL OUTLIER CLIPPING (WINSORIZATION)
    # =====================================================================
    clipping_limit = 3.5
    
    x_train_clipped = torch.clamp(x_train_std, min=-clipping_limit, max=clipping_limit)
    x_eval_clipped = torch.clamp(x_eval_std, min=-clipping_limit, max=clipping_limit)
    
    price_std_target_train_tensor = torch.clamp(y_train_std, min=-clipping_limit, max=clipping_limit)
    price_std_target_eval_tensor = torch.clamp(y_eval_std, min=-clipping_limit, max=clipping_limit)

    # =====================================================================
    # 6. FORCE 3D EXPANSION FOR TIME-SERIES LSTM INPUT PARITY
    # =====================================================================
    # Transforms shape from [Batch, Steps] to [Batch, Steps, Feature=1]
    price_std_train_tensor = x_train_clipped.unsqueeze(-1)
    price_std_eval_tensor = x_eval_clipped.unsqueeze(-1)

    # --- LOCALIZED SANDBOX REPORTING LOGS ---
    print("📊 DIAGNÓSTICO DE PADRONIZAÇÃO LOCAL COM CLIPPING DE OUTLIERS (PRICE MODE):")
    print("-" * 85)
    print(f"   • Formato Train Features : {list(price_std_train_tensor.shape)} [Batch, Timesteps, Feature]")
    print(f"   • Formato Train Targets  : {list(price_std_target_train_tensor.shape)} [Batch, Target]")
    print(f"   • Formato Eval Features  : {list(price_std_eval_tensor.shape)}")
    print(f"   • Formato Eval Targets   : {list(price_std_target_eval_tensor.shape)}")
    print("-" * 85)
    print(f"   • Stock 0 Train - Média Ativo  : {price_std_train_tensor[0].mean().item():.4f} (Deve ser ~0)")
    print(f"   • Stock 0 Train - Desvio Ativo : {price_std_train_tensor[0].std().item():.4f} (Deve ser ~1)")
    print("-" * 85)
else:
    print("❌ Erro de Sequência: DataFrame 'df' não localizado na memória.")

In [ ]:
#### [AI-CONTEXT]
#### ID: 014
#### ROLE: Prepare a pile of finance benchmark tensors to be processed along with the scaled price tensor in the Time-Series LSTM Branch.
#### INPUT: Generate standardized tensors (successful execution of ID: 013).
#### OUTPUT: Fetch market data aligned strictly with the last 61 Brazilian business days (B3 calendar). Group the benchmarks by inherent financial properties and apply a hybrid normalization strategy.


import yfinance as yf
import torch
import numpy as np
import pandas as pd
from bcb import sgs

# (Steps 1 through 4 remain identical to your clean collection/alignment setup)
yahoo_indicators = {
    "GOLD": "GC=F", "USD_BRL": "USDBRL=X", "IBOV": "^BVSP", "US2YT": "2YY=F",
    "EWZ": "EWZ", "IBX50": "^IBX50", "US500": "^GSPC", "BTC_USD": "BTC-USD",
    "IMAB": "IMAB11.SA", "WTI": "CL=F", "VIX": "^VIX", "IFIX": "XFIX11.SA"
}
bcb_indicators = {"CDI": 12, "IPCA": 433}

anchor_ticker = "^BVSP"
ibov_df = yf.download(anchor_ticker, period="150d", progress=False)
br_business_days = ibov_df['Close'][anchor_ticker].dropna().index[-61:] if isinstance(ibov_df.columns, pd.MultiIndex) else ibov_df['Close'].dropna().index[-61:]

aligned_data = {}
for name, ticker in yahoo_indicators.items():
    raw_df = yf.download(ticker, period="150d", progress=False)
    close_series = raw_df['Close'][ticker] if isinstance(raw_df.columns, pd.MultiIndex) else raw_df['Close']
    aligned_data[name] = close_series.reindex(br_business_days).ffill().bfill().values

start_date = (br_business_days[0] - pd.Timedelta(days=45)).strftime('%Y-%m-%d')
end_date = br_business_days[-1].strftime('%Y-%m-%d')
for name, sgs_code in bcb_indicators.items():
    raw_bcb = sgs.get(sgs_code, start=start_date, end=end_date)
    if isinstance(raw_bcb, pd.DataFrame): raw_bcb = raw_bcb.iloc[:, 0]
    if sgs_code == 433:
        daily_filled = raw_bcb.reindex(pd.date_range(start=start_date, end=end_date, freq='D')).ffill()
        aligned_series = daily_filled.reindex(br_business_days).bfill()
    else:
        aligned_series = raw_bcb.reindex(br_business_days).ffill().bfill()
    aligned_data[name] = aligned_series.values

# =====================================================================
# 5. FIXED: UNIFIED MULTI-CHANNEL SCALING & LEAK-FREE TRANSFORMS
# =====================================================================
processed_vectors = []
eps = 1e-6

for name, raw_array in aligned_data.items():
    arr_f32 = raw_array.astype(np.float32)
    
    # FIXED: Calculate mean/std strictly over the first 60 historical days to prevent leakage
    series_mean = np.mean(arr_f32[:-1])
    series_std = np.std(arr_f32[:-1])
    
    # Scale all 61 days using the lookback-only parameters (Preserves time step alignment!)
    equalized_series = (arr_f32 - series_mean) / (series_std + eps)
    
    # Symmetrical Clipping
    clipped_series = np.clip(equalized_series, -3.5, 3.5)
    processed_vectors.append(clipped_series)

# Combine into a clean 2D Matrix grid layout: [Timesteps=61, Total_Features=14]
market_matrix_2d = np.column_stack(processed_vectors)
market_base_tensor = torch.tensor(market_matrix_2d, dtype=torch.float32)

# =====================================================================
# 6. EXPAND CHANNELS INTO BROADCASTED COPIES MATCHING ID 013 SPLITS
# =====================================================================
# Extracts actual active row constraints from your price_std variables dynamically
num_train_stocks = price_std_train_tensor.shape[0] 
num_eval_stocks = price_std_eval_tensor.shape[0]   

# .unsqueeze(0) makes it, .expand() replicates it to [Stocks, 61, 14]
benchmark_train_tensor = market_base_tensor.unsqueeze(0).expand(num_train_stocks, -1, -1)
benchmark_eval_tensor = market_base_tensor.unsqueeze(0).expand(num_eval_stocks, -1, -1)

# =====================================================================
# VERIFICATION PHASE
# =====================================================================
print(f"✅ Total Indicators Stacked into Unified Matrix: {market_base_tensor.shape[1]}")
print("-" * 85)
print(f"   • Unified 2D Base Market Tensor Shape : {list(market_base_tensor.shape)} [Timesteps, Indicators]")
print(f"   • Broadcasted Train Benchmark Shape   : {list(benchmark_train_tensor.shape)} [Stocks, Timesteps, Indicators]")
print(f"   • Broadcasted Eval Benchmark Shape    : {list(benchmark_eval_tensor.shape)} [Stocks, Timesteps, Indicators]")
print("-" * 85)
print(f"   • Global Macro-Tensor Memory Layout Matrix Validated. Ready for Fusion.")
print("-" * 85)

In [ ]:
#### [AI-CONTEXT]
#### ID: 015
#### ROLE: implement the 3D Master Tensor Layout logic. It takes the price_std_train_tensor and the 14 indicators from normalized_benchmarks, scales them symmetrically, and stacks them into the final (train batch, 60, 15) train tensor.
#### INPUT: standardized tensors and rescaled_tensor.
#### OUTPUT: master_train_3d_tensor and master_input_train_3d_tensor


import torch

# =====================================================================
# 3D MASTER TRAIN TENSOR LAYOUT CONSTRUCTION (PRICE MODE)
# =====================================================================

# Step 1 & 2: Fuse the 3D standardized price tensor with the 3D broadcasted benchmark tensor
# price_std_train_tensor shape: [1116, 60, 1]
# FIXED: Sliced benchmark_train_tensor to [:, :-1, :] to match the time axis perfectly!
master_train_3d_tensor = torch.cat([
    price_std_train_tensor, 
    benchmark_train_tensor[:, :-1, :]
], dim=2)

# Step 3: Assign directly to your master input tracking variable
master_input_train_3d_tensor = master_train_3d_tensor

# =====================================================================
# VERIFICATION PHASE
# =====================================================================
print("✅ [3D MASTER TRAIN TENSOR LAYOUT VERIFIED - PRICE MODE COMPLETE]")
print("=" * 70)
print(f"Master Train Tensor Shape: {list(master_input_train_3d_tensor.shape)} (Expected: [batch, 60, 15])")
print(f"Dimension 0 (Stocks)     : {master_input_train_3d_tensor.shape[0]}")
print(f"Dimension 1 (Days)       : {master_input_train_3d_tensor.shape[1]} [Price Wave Steps]")
print(f"Dimension 2 (Features)   : {master_input_train_3d_tensor.shape[2]}")
print("-" * 70)

# Quick validation check on a vector slice
sample_vector = master_input_train_3d_tensor[0, 0, :]
print(f"Sample Feature Vector (Stock 0, Day 0) Shape : {list(sample_vector.shape)}")
print(f"Primary Anchor (Stock Price Wave) Value      : {sample_vector[0].item():.4f}")
print(f"First Macro Indicator (GOLD) Value           : {sample_vector[1].item():.4f}")
print("=" * 70)

In [ ]:
#### [AI-CONTEXT]
#### ID: 016
#### ROLE: implement the 3D Master Tensor Layout logic. It takes the price_std_eval_tensor and the 14 indicators from minmax_scaled_tensors, scales them symmetrically, and stacks them into the final (eval batch, 61, 15) eval tensor.
#### INPUT: standardized tensors and rescaled_tensor.
#### OUTPUT: master_eval_3d_tensor and master_input_eval_3d_tensor


import torch

# =====================================================================
# 3D MASTER EVAL TENSOR LAYOUT CONSTRUCTION (PRICE MODE)
# =====================================================================

# Step 1 & 2: Fuse the 3D evaluation price tensor with the 3D broadcasted evaluation benchmarks
# price_std_eval_tensor shape: [batch, 60, 1]
# FIXED: Sliced benchmark_eval_tensor to [:, :-1, :] to match the time axis perfectly!
master_eval_3d_tensor = torch.cat([
    price_std_eval_tensor, 
    benchmark_eval_tensor[:, :-1, :]
], dim=2)

# Step 3: Assign directly to your master evaluation input tracking variable
master_input_eval_3d_tensor = master_eval_3d_tensor

# =====================================================================
# VERIFICATION PHASE
# =====================================================================
print("✅ [3D MASTER EVAL TENSOR LAYOUT VERIFIED - PRICE MODE COMPLETE]")
print("=" * 70)
print(f"Master Input Eval Tensor Shape: {list(master_input_eval_3d_tensor.shape)} (Expected: [batch, 60, 15])")
print(f"Dimension 0 (Stocks)          : {master_input_eval_3d_tensor.shape[0]}")
print(f"Dimension 1 (Days)            : {master_input_eval_3d_tensor.shape[1]} [Price Wave Steps]")
print(f"Dimension 2 (Features)        : {master_input_eval_3d_tensor.shape[2]}")
print("-" * 70)

# Quick validation check on a vector slice
sample_vector = master_input_eval_3d_tensor[0, 0, :]
print(f"Sample Feature Vector (Stock 0, Day 0) Shape : {list(sample_vector.shape)}")
print(f"Primary Anchor (Stock Price Wave) Value      : {sample_vector[0].item():.4f}")
print(f"First Macro Indicator (GOLD) Value           : {sample_vector[1].item():.4f}")
print("=" * 70)

In [ ]:
#### [AI-CONTEXT]
#### ID: 017
#### ROLE: Determine the hyperparameters for the Multi-Layer Perceptron.
#### INPUT: master_3d_tensor and details_input_tensor.
#### OUTPUT: Hyperparameters.


# =====================================================================
# MULTIMODAL MLP-LSTM-ATTENTION HYPERPARAMETERS (PRICE MODE)
# =====================================================================

# --- 1. Global Vocabulary & Sequence Configurations ---
vocab_size = len(tokenizer)                                   # FinBERT vocabulary size bounds

# Programmatically track text sequence length from the 2D tensor shape safely
raw_details_length = details_input_tensor.shape               

# Target index 1 to capture the exact token sequence depth as a clean integer
details_length = int(raw_details_length[1])

# --- 2. Categorical / NLP Feature Processing Branch ---
embedding_dim = 64                                            # Latent embedding depth for text tokens
output_dim = 32                                               # Linear projection output depth of NLP features

# --- 3. Static Tabular Feature Branch ---
tabular_features_dim = 1                                      # 1 Feature (LIQUIDITY)
tabular_output_dim = 8                                        # Linear projection output depth of Tabular features

# --- 4. Time-Series Recurrent Branch ---
lstm_input_size = 15                                          # 1 Stock Price + 14 Macro Benchmarks
num_layers = 2                                                # Stacked deep recurrent layers
lstm_hidden_size = 128                                        # Hidden units inside each LSTM cell block

# Programmatically locks your time lookback sequence length to exactly 60 days
time_sequence_length = master_input_train_3d_tensor.shape[1]

# --- 5. Multi-Head Attention & Regularization Blocks ---
dropout = 0.2                                                 # Standard dropout ratio across deep layers
num_heads = 8                                                 # Number of parallel attention heads

# --- 6. Downstream Fusion Layer Math Validation ---
# Combines processing outputs from NLP Branch, Tabular Branch, and LSTM Time-Series Branch
fused_feature_dim = output_dim + tabular_output_dim + lstm_hidden_size  # 32 + 8 + 128 = 168 features

# --- 7. Deep Feed-Forward Residual Block Configuration ---
ffn_input_dim = fused_feature_dim                            # Input dimensionality mapping (168 channels)
ffn_hidden_dim = 512                                          # Hidden node layer expansion constraint

print("⚙️ HYPERPARAMETER ARCHITECTURE DIAGNOSTIC (PRICE MODE ACTIVATED):")
print("-" * 65)
print(f"   • FinBERT Token Vocab Size     : {vocab_size}")
print(f"   • Text Input Token Length      : {details_length} (Tokens per Asset)")
print(f"   • Tabular Feature Input Dims   : {tabular_features_dim} -> Output Dims: {tabular_output_dim}")
print(f"   • LSTM Input Sequence Length   : {time_sequence_length} Days of Prices")
print(f"   • Attention Fused Dimension    : {fused_feature_dim} features")
print(f"   • FFN Layer Dimensions         : Input: {ffn_input_dim} -> Hidden: {ffn_hidden_dim}")
print("-" * 65)

if fused_feature_dim % num_heads == 0:
    print("   ✅ Structural Match: Fused dimension is perfectly divisible by num_heads.")
else:
    print(f"   ❌ Dimension Conflict: Fused dimension ({fused_feature_dim}) cannot be split into {num_heads} heads!")
print("-" * 65)

In [ ]:
#### [AI-CONTEXT]
#### ID: 018
#### ROLE: Define the Multi-Head Attention block.
#### INPUT: Hyperparameters in script ID: 017.
#### OUTPUT: AttentionBlock() module.


import torch
import torch.nn as nn

class AttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        # Enforce strict multi-head attention structural divisibility check at construction time
        assert embed_dim % num_heads == 0, f"❌ embed_dim ({embed_dim}) must be perfectly divisible by num_heads ({num_heads})"
        
        # Multi-Head Attention layer
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        
        # Feed-Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
            nn.Dropout(dropout)
        )
        
        self.layernorm1 = nn.LayerNorm(embed_dim)
        self.layernorm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        # x shape expected: (batch_size, sequence_length, embed_dim)
        
        # 1. Attention + Residual
        attn_output, _ = self.mha(x, x, x)
        x = self.layernorm1(x + attn_output)
        
        # 2. Feed-Forward + Residual
        ffn_output = self.ffn(x)
        x = self.layernorm2(x + ffn_output)
        return x

# --- SANITY CHECK DIAGNOSTIC LOOP (60-DAY PRICE TIMELINE) ---
if __name__ == "__main__":
    # Simulate a training batch using your true 60-day window and 160 fused features
    sample_input = torch.randn(32, time_sequence_length, fused_feature_dim)
    
    # Instantiate the module using your exact global hyperparameters from ID: 017
    block = AttentionBlock(embed_dim=fused_feature_dim, num_heads=num_heads, dropout=dropout)
    sample_output = block(sample_input)
    
    print("=" * 75)
    print("✅ [ATTENTION BLOCK RE-VERIFIED FOR THE 60-DAY PRICE WINDOW]")
    print("=" * 75)
    print(f"   • Mock Input Dimensional Shape  : {list(sample_input.shape)} (Expected: [32, 60, 160])")
    print(f"   • Attention Output Tensor Shape : {list(sample_output.shape)}")
    print("=" * 75)

In [ ]:
#### [AI-CONTEXT]
#### ID: 019
#### ROLE: Define a deep, modular Feed-Forward network component with residual skip connections.
#### INPUT: Fused feature outputs from structural model branches.
#### OUTPUT: DeepFeedForwardBlock() module.


import torch
import torch.nn as nn

class DeepFeedForwardBlock(nn.Module):
    """
    A robust, deep Feed-Forward Network block equipped with Layer Normalization, 
    SiLU (Swish) activations, and residual skip connections to process dense financial contexts.
    """
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.2):
        super().__init__()
        
        # First transformation layer
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.activation = nn.SiLU() # SiLU/Swish generally outperforms ReLU in deep financial nets
        self.dropout = nn.Dropout(dropout_rate)
        
        # Second projection layer to bring dimensions back for the skip connection
        self.linear2 = nn.Linear(hidden_dim, input_dim)
        self.norm2 = nn.LayerNorm(input_dim)
        
    def forward(self, x):
        # Store identity context matrix for the residual shortcut link
        residual = x
        
        # Process through the primary feed-forward path
        out = self.linear1(x)
        out = self.norm1(out)
        out = self.activation(out)
        out = self.dropout(out)
        
        out = self.linear2(out)
        out = self.norm2(out)
        
        # Apply the residual addition (Skip Connection) to avoid vanishing gradients
        return self.activation(out + residual)

# --- SANITY CHECK DIAGNOSTIC LOOP (FUSED FEATURE SPACE) ---
if __name__ == "__main__":
    # Simulate an active batch using your global 168 fused multimodal features dimension
    fused_features_mock = torch.randn(32, 168)
    
    # Initialize the FFN block using a standard 512 hidden expansion layer
    ffn_module = DeepFeedForwardBlock(input_dim=168, hidden_dim=512, dropout_rate=0.2)
    output_tensor = ffn_module(fused_features_mock)
    
    print("=" * 70)
    print("✅ [ID: 021B DEEP FEED-FORWARD RESIDUAL BLOCK VALIDATED]")
    print("=" * 70)
    print(f"• Input Structural Shape  : {list(fused_features_mock.shape)} (Expected: [32, 168])")
    print(f"• FFN Output Tensor Shape : {list(output_tensor.shape)} (Dimensions preserved!)")
    print("=" * 70)


In [ ]:
#### [AI-CONTEXT]
#### ID: 020
#### ROLE: Implement the Tabular Branch.
#### INPUT: liquidity_train_tensor, liquidity_eval_tensor, and Hyperparameters.
#### OUTPUT: TabularBranch() module.


import torch
import torch.nn as nn

class TabularBranch(nn.Module):
    def __init__(self, tabular_features_dim, tabular_output_dim):
        super().__init__()
        # 1. Project the static inputs to an intermediate hidden representation space
        self.fc1 = nn.Linear(tabular_features_dim, 16)
        
        # 2. BatchNorm1d implementation preserves contrast across batch metrics
        self.ln1 = nn.BatchNorm1d(16)
        
        # 3. Activation layer and final structural feature map compression projection
        self.relu = nn.LeakyReLU(0.1)
        self.fc2 = nn.Linear(16, tabular_output_dim)

    def forward(self, tabular_input):
        # Defensively ensure inputs are formatted as 2D column vectors: [Batch Size, Features]
        if tabular_input.dim() == 1:
            tabular_input = tabular_input.unsqueeze(1)

        # Process inputs through dense network layers
        x = self.fc1(tabular_input)
        x = self.ln1(x)
        x = self.relu(x)
        return self.fc2(x)

# --- SANITY CHECK DIAGNOSTIC LOOP (61-DAY PIPELINE VALIDATION) ---
if __name__ == "__main__":
    # Simulate a standardized batch load of 32 stocks with 1 liquidity feature
    mock_tabular_data = torch.randn(32, 1)

    # Initialize the branch with 1 feature input and an 8-dimensional projection space
    tab_module = TabularBranch(tabular_features_dim=1, tabular_output_dim=8)

    # Run the test inference pass
    mock_output = tab_module(mock_tabular_data)

    print("=" * 75)
    print("✅ [TABULAR PROCESSING BRANCH MODULE CLEANED AND VERIFIED]")
    print("=" * 75)
    print(f"   • Mock Tabular Feature Input Shape    : {list(mock_tabular_data.shape)}")
    print(f"   • Tabular Output Projections Vector   : {list(mock_output.shape)}")
    print("=" * 75)

In [ ]:
#### [AI-CONTEXT]
#### ID: 021
#### ROLE: Implement the Time-Series LSTM Branch.
#### INPUT: price_train_tensor, price_eval_tensor, and Hyperparameters.
#### OUTPUT: StockAdaptivepriceabilityBranch() and StockAdaptivepriceabilityEvalBranch() modules.


import torch
import torch.nn as nn

class StockAdaptivePriceabilityBranch(nn.Module):
    def __init__(self, lstm_input_size, lstm_hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm_input_size = lstm_input_size
        
        # 1. Feature Attention Gating System
        # Learns to weight the 1 stock price channel and 14 macro benchmarks dynamically per step
        self.feature_attention = nn.Sequential(
            nn.Linear(lstm_input_size, lstm_input_size),
            nn.LeakyReLU(0.1),
            nn.Linear(lstm_input_size, lstm_input_size),
            nn.Sigmoid() 
        )
        
        # 2. Standard Temporal Sequence Branch
        self.lstm = nn.LSTM(
            input_size=lstm_input_size, 
            hidden_size=lstm_hidden_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # 3. FIXED: Aligned projection depth with your true lstm_hidden_size variable (128)
        # Keeps hidden states dimensionally consistent for downstream master fusion steps!
        self.fc = nn.Linear(lstm_hidden_size, lstm_hidden_size)
        self.activation = nn.LeakyReLU(0.1)

    def forward(self, x):
        # x shape expected under price mode: (batch_size, 60, 15)
        batch_size, seq_len, num_features = x.shape
        
        # Flatten the temporal axis to compute step-wise feature gating masks
        x_flat = x.contiguous().view(-1, num_features)
        weights = self.feature_attention(x_flat)
        masked_features = x_flat * weights
        
        # Reconstruct the 3D tensor back for the recurrent layer
        lstm_input = masked_features.view(batch_size, seq_len, num_features)
        
        # Capture full lstm_out sequence history across all 60 absolute price steps
        # lstm_out shape: (batch_size, 60, lstm_hidden_size)
        lstm_out, _ = self.lstm(lstm_input)
        
        # Linearly project each day's features from 128 to 128 dimensions to preserve context
        # Output shape: (batch_size, 60, 128) -> Perfectly matched for your AttentionBlock fusion!
        out = self.fc(lstm_out)
        return self.activation(out)

# --- SANITY CHECK DIAGNOSTIC LOOP (60-DAY TIMELINE WINDOW) ---
if __name__ == "__main__":
    # Simulate an active training batch using your exact global parameters from ID: 017
    sample_batch = torch.randn(32, 60, 15)
    
    # Initialize the unified processing module instance using your global settings
    model_branch = StockAdaptivePriceabilityBranch(
        lstm_input_size=15, 
        lstm_hidden_size=128, 
        num_layers=2, 
        dropout=0.2
    )
    
    # Verify training mode execution pass
    model_branch.train()
    train_output = model_branch(sample_batch)
    
    # Verify evaluation mode execution pass using the same class instance
    model_branch.eval()
    with torch.no_grad():
        eval_output = model_branch(sample_batch)
        
    print("=" * 75)
    print("✅ [UNIFIED TIME-SERIES LSTM BRANCH VERIFIED FOR PRICE SEQUENCES]")
    print("=" * 75)
    print(f"   • Input Tensor Base Price Footprint : {list(sample_batch.shape)}")
    print(f"   • Train Pass Output Shape (3D)      : {list(train_output.shape)}")
    print(f"   • Eval Pass Output Shape  (3D)      : {list(eval_output.shape)}")
    print("=" * 75)

In [ ]:
#### [AI-CONTEXT]
#### ID: 022
#### ROLE: Implement the Categorical/NLP Branch.
#### INPUT: details_train_tensor, details_eval_tensor, and Hyperparameters.
#### OUTPUT: DetailsBranchTransformer() module.


import torch
import torch.nn as nn

class DetailsBranchTransformer(nn.Module):
    def __init__(self, vocab_size, details_length, embedding_dim, num_heads, dropout, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Max boundary sequence length parameter (handles the largest theoretical padding window)
        # We ensure it's initialized with a large enough integer space (e.g. 512 or higher)
        max_len = max(512, int(details_length))
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, embedding_dim))
        
        # Stacking TWO Attention Blocks
        self.transformer_blocks = nn.Sequential(
            AttentionBlock(embedding_dim, num_heads, dropout),
            AttentionBlock(embedding_dim, num_heads, dropout)
        )
        
        self.fc = nn.Linear(embedding_dim, output_dim)
        self.activation = nn.LeakyReLU(0.1)

    def forward(self, x):
        # x shape dynamically changes based on batch padding: (batch_size, current_seq_len)
        batch_size, seq_len = x.shape
        
        # 1. FIX: Dynamically slice the positional matrix to match this specific batch's sequence length
        positioned_embeds = self.embedding(x) + self.pos_embedding[:, :seq_len, :]
        
        # 2. Pass through the stack of 2 custom Attention Blocks
        attn_out = self.transformer_blocks(positioned_embeds)
        
        # 3. Global Average Pooling along the token timeline dimension (dim=1)
        x_pooled = attn_out.mean(dim=1)
        
        # 4. Linearly project down to the final target dimension (output_dim = 32)
        return self.activation(self.fc(x_pooled))

# --- SANITY CHECK DIAGNOSTIC LOOP ---
if __name__ == "__main__":
    # Simulate a training batch of 32 assets across 384 text tokens
    sample_tokens = torch.randint(0, 30522, (32, 384)) 
    
    # Initialize the categorical tracking branch
    nlp_branch = DetailsBranchTransformer(
        vocab_size=30522,
        details_length=384,
        embedding_dim=64,
        num_heads=8,
        dropout=0.2,
        output_dim=32
    )
    
    output = nlp_branch(sample_tokens)
    
    print("=" * 75)
    print("✅ [CATEGORICAL/NLP TRANSFORMER BRANCH ARCHITECTURE VERIFIED]")
    print("=" * 75)
    print(f"   • Text Tokens Input Footprint: {list(sample_tokens.shape)}")
    print(f"   • Output Feature Vector Shape: {list(output.shape)} (Expected: [32, 32])")
    print("=" * 75)

In [ ]:
#### [AI-CONTEXT]
#### ID: 023
#### ROLE: Implement the Multi-Layer Perceptron Architecture.
#### INPUT: StockAdaptivePriceabilityBranch, DetailsBranchTransformer, DeepFeedForwardBlock, and hyperparameters.
#### OUTPUT: StockProfitabilityPredictor() module.


import torch
import torch.nn as nn

class StockProfitabilityPredictor(nn.Module):
    def __init__(self, vocab_size, details_length, embedding_dim, output_dim, num_heads,
                 tabular_features_dim, tabular_output_dim,
                 lstm_input_size, lstm_hidden_size, num_layers, dropout,
                 ffn_input_dim, ffn_hidden_dim):
        super().__init__()
        
        # 1. NLP Transformer Feature Branch (Outputs output_dim = 32 features)
        self.details_branch = DetailsBranchTransformer(
            vocab_size=vocab_size, 
            details_length=details_length, 
            embedding_dim=embedding_dim, 
            num_heads=num_heads,
            dropout=dropout,
            output_dim=output_dim
        )
        
        # 2. Static Tabular Feature Branch (Outputs tabular_output_dim = 8 features)
        self.tabular_branch = TabularBranch(
            tabular_features_dim=tabular_features_dim,
            tabular_output_dim=tabular_output_dim
        )
        
        # 3. Time-Series Gated LSTM Branch (Outputs 128 hidden features per step over 60 days)
        self.price_branch = StockAdaptivePriceabilityBranch(
            lstm_input_size=lstm_input_size, 
            lstm_hidden_size=lstm_hidden_size, 
            num_layers=num_layers, 
            dropout=dropout
        )
        
        # 4. Custom Transformer Attention Block (From ID: 017)
        self.fused_feature_dim = output_dim + tabular_output_dim + lstm_hidden_size
        self.fused_attention = AttentionBlock(
            embed_dim=self.fused_feature_dim,
            num_heads=num_heads,  
            dropout=dropout
        )
        
        # 5. Deep Feed-Forward Residual Block Layer Insertion
        self.deep_ffn = DeepFeedForwardBlock(
            input_dim=ffn_input_dim, 
            hidden_dim=ffn_hidden_dim, 
            dropout_rate=dropout
        )

        # 6. Final Target Linear Projection Head
        self.final_projection = nn.Linear(self.fused_feature_dim, 1) 
        
        # Apply custom xavier weight initialization to prevent gradient instabilities
        self.apply(self._init_weights)

    def forward(self, details_tensor, tabular_tensor, price_tensor):
        # details_tensor shape: (batch_size, details_length)
        # tabular_tensor shape: (batch_size, tabular_features_dim)
        # price_tensor shape:   (batch_size, 60, 15) [Absolute Price Mode]
        
        # Branch A: Process text tokens through the NLP Transformer -> Shape: (batch_size, 32)
        d_feat = self.details_branch(details_tensor)
        
        # Branch B: Process static metrics through the Tabular Branch -> Shape: (batch_size, 8)
        t_feat = self.tabular_branch(tabular_tensor)
        
        # Branch C: Process time-series data through the LSTM layer -> Shape: (batch_size, 60, 128)
        p_seq_feat = self.price_branch(price_tensor)
        
        # Combine non-sequential structural projections (NLP + Tabular) -> Shape: (batch_size, 32 + 8 = 40)
        static_feat = torch.cat((d_feat, t_feat), dim=1)
        
        # Expand the flat 2D static features across the 60-day timeline axis to enable sequential fusion
        static_feat_expanded = static_feat.unsqueeze(1).expand(-1, p_seq_feat.size(1), -1)
        
        # Multimodal Fusion: Concatenate static features with sequential market features -> Shape: (batch_size, 60, 168)
        combined_sequence = torch.cat((static_feat_expanded, p_seq_feat), dim=2)
        
        # Pass the 168-channel fused sequence into the Multi-Head Attention block -> Shape: (batch_size, 60, 168)
        attn_fused_feat = self.fused_attention(combined_sequence)
        
        # Compress the timeline sequence into a fixed vector using Global Average Pooling -> Shape: (batch_size, 168)
        pooled_context = attn_fused_feat.mean(dim=1)

        # Deep FFN refines the multi-branch interactions safely -> Shape: (batch_size, 168)
        refined_context = self.deep_ffn(pooled_context) 
        
        # Final target prediction -> Shape: (batch_size, 1) [Forecasted Day 61 Absolute Price Z-score]
        return self.final_projection(refined_context) 

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                m.bias.data.fill_(0.01)

# --- SANITY CHECK DIAGNOSTIC LOOP (60-DAY PRICE WINDOW) ---
if __name__ == "__main__":
    # Simulate an active training batch of 32 stocks matching global limits
    sample_text_tokens = torch.randint(0, 30522, (32, 478))
    sample_tabular_data = torch.randn(32, 1)
    sample_price_sequences = torch.randn(32, 60, 15)
    
    # Initialize the complete unified core model architecture using your updated constructor signature
    model = StockProfitabilityPredictor(
        vocab_size=30522, details_length=478, embedding_dim=64, output_dim=32, num_heads=8,
        tabular_features_dim=1, tabular_output_dim=8,
        lstm_input_size=15, lstm_hidden_size=128, num_layers=2, dropout=0.2,
        ffn_input_dim=168, ffn_hidden_dim=512
    )
    
    # Verify training state execution pass
    model.train()
    train_pred = model(sample_text_tokens, sample_tabular_data, sample_price_sequences)
    
    # Verify evaluation state execution pass using the exact same class object
    model.eval()
    with torch.no_grad():
        eval_pred = model(sample_text_tokens, sample_tabular_data, sample_price_sequences)
        
    print("=" * 75)
    print("✅ [UNIFIED TRIPLE-BRANCH MULTIMODAL CORE ARCHITECTURE REGISTERED]")
    print("=" * 75)
    print(f"   • Train Pass Batch Prediction Output Shape : {list(train_pred.shape)} (Expected: [32, 1])")
    print(f"   • Eval Pass Batch Prediction Output Shape  : {list(eval_pred.shape)}")
    print("=" * 75)

In [ ]:
#### [AI-CONTEXT]
#### ID: 024
#### ROLE: Implement a multi-modal batch training system.
#### INPUT: details_input_tensor, master_input_train_3d_tensor, price_std_target_train_tensor, and StockProfitabilityPredictor.
#### OUTPUT: Train the model with the dataset.


import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 1. CUSTOM DATASET WRAPPER (UPDATED FOR TRIPLE BRANCH)
# =====================================================================
class FundDataset(Dataset):
    def __init__(self, details, tabular, price, targets):
        self.details = details
        self.tabular = tabular
        self.price = price
        self.targets = targets 
        
    def __len__(self):
        return len(self.targets)
        
    def __getitem__(self, idx):
        return (self.details[idx], self.tabular[idx], self.price[idx], self.targets[idx])

# 'liquidity_train_tensor' from ID: 009 is passed as the second structural input
dataset = FundDataset(
    details_train_tensor, 
    liquidity_train_tensor, 
    master_input_train_3d_tensor, 
    price_std_target_train_tensor
)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# =====================================================================
# 2. INITIALIZE UNIFIED HARDWARE ENVIRONMENT & MODEL
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Appended ffn_input_dim and ffn_hidden_dim parameters to match updated ID: 022 class signature
model = StockProfitabilityPredictor(
    vocab_size=vocab_size,
    details_length=details_length,
    embedding_dim=embedding_dim,
    output_dim=output_dim,
    num_heads=num_heads,
    tabular_features_dim=tabular_features_dim,
    tabular_output_dim=tabular_output_dim,
    lstm_input_size=lstm_input_size,
    lstm_hidden_size=lstm_hidden_size,  
    num_layers=num_layers,
    dropout=dropout,
    ffn_input_dim=ffn_input_dim,
    ffn_hidden_dim=ffn_hidden_dim
).to(device)

epochs = 100  # Reduced from 120 to save compute and eliminate training plateaus
max_lr = 1e-3  # Slightly higher peak learning rate to explore early gradients

# Added explicit weight decay to prevent overfitting on long descriptions
optimizer = optim.AdamW(model.parameters(), lr=max_lr / 25.0, weight_decay=1e-3)

# Replaced CosineAnnealing with OneCycleLR for super-convergence
scheduler = OneCycleLR(
    optimizer,
    max_lr=max_lr,
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    pct_start=0.2,  # Spends 20% of training warming up, then decays down
    div_factor=25.0,
    final_div_factor=1e4
)

# Swapped plain MSELoss for Huber Loss to make model resilient to market anomalies
criterion = nn.HuberLoss(delta=1.0)

# --- 3. STORAGE & MONITORING KEYS ---
iteration_losses = []
iteration_indices = []
batch_print_frequency = 5 
global_step = 0

# =====================================================================
# 4. ACTIVE TRAINING LOOP
# =====================================================================
print(f"🚀 Training initiated on device: {device} | Horizon: {epochs} epochs.")
print(f"📊 Dataset check: Running {len(dataset)} items across {len(train_loader)} batches.\n")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    valid_batches = 0
    
    for batch in train_loader:
        details, tabular, price, targets = batch
        
        # Move all incoming batch tensors to the active compute unit
        details = details.long().to(device)     
        tabular = tabular.float().to(device)
        price = price.float().to(device)      
        targets = targets.float().to(device)    
        
        optimizer.zero_grad()
        # Feed the required tabular features tensor to the forward step execution call
        outputs = model(details, tabular, price)
        
        # Calculate standard mean squared error loss trajectory
        loss = criterion(outputs.view(-1), targets.view(-1))
        
        if torch.isnan(loss):
            print(f"⚠️ Warning: NaN detected at Epoch {epoch+1}, skipping batch.")
            continue
            
        loss.backward()
        
        # Clip gradient distributions to safeguard the deep recurrent hidden layers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        valid_batches += 1
        global_step += 1
        
        # Track mini-batch performance over training step sequences
        if valid_batches % batch_print_frequency == 0:
            iteration_losses.append(loss.item())
            iteration_indices.append(global_step)
            
    if valid_batches > 0:
        scheduler.step()

        print_interval = max(1, epochs // 10)
        if epoch == 0 or (epoch + 1) % print_interval == 0:
            avg_loss = epoch_loss / valid_batches
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1:02d}/{epochs} | Avg Loss: {avg_loss:.6f} | LR: {current_lr:.6e}")

# =====================================================================
# 5. HARDWARE CACHE PURGE REMNANTS
# =====================================================================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache() 

print(f"\n✅ Training loop is complete. Next steps can be executed now.\n")

In [ ]:
#### [AI-CONTEXT]
#### ID: 025
#### ROLE: Visually analyze your training convergence and check for gradient stability over time.
#### INPUT: Complete train process (successful execution of ID: 024).
#### OUTPUT: Plot a chart for the loss over the epochs.


import matplotlib.pyplot as plt

# Check if we successfully gathered mini-batch metric points from ID 022
if len(iteration_losses) > 0:
    plt.figure(figsize=(10, 5))
    plt.plot(iteration_indices, iteration_losses, color='darkorange', linewidth=2, label='MSE Loss')
    plt.title('Multi-Modal Stock Predictor - Training Progress', fontsize=12, fontweight='bold')
    plt.xlabel('Global Mini-Batch Iteration Step')
    plt.ylabel('Loss Value')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.show()
else:
    print("⚠️ No iteration data logged. Try increasing your epoch limits or reducing the print frequency flag.")

In [ ]:
#### [AI-CONTEXT]
#### ID: 026
#### ROLE: Implement the evaluation process and collect telemetry metrics about the model.
#### INPUT: StockProfitabilityPredictorEval(), master_input_eval_3d_tensor, price_std_target_eval_tensor. Complete train process (successful execution of ID: 024).
#### OUTPUT: Evaluate the model with Mean Absolute Error (MAE), R-squared, and 0.1 standard deviation Tolerance Accuracy.


import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_absolute_error

# =====================================================================
# 1. OUT-OF-SAMPLE DATA CONTAINER PACKAGING (UPDATED FOR TRIPLE BRANCH)
# =====================================================================
class EvalFundDataset(Dataset):
    def __init__(self, details, tabular, price, targets):
        self.details = details
        self.tabular = tabular
        self.price = price
        self.targets = targets 
        
    def __len__(self):
        return len(self.targets)
        
    def __getitem__(self, idx):
        return (self.details[idx], self.tabular[idx], self.price[idx], self.targets[idx])

# Instantiate validation dataset using evaluation arrays alongside liquidity features from ID 009
val_ds = EvalFundDataset(
    details_eval_tensor, 
    liquidity_eval_tensor, # Integrated missing tabular dimension
    master_input_eval_3d_tensor, 
    price_std_target_eval_tensor
)

# Shuffle=False is mandatory for evaluation tasks to preserve mapping orders
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

# =====================================================================
# 2. STANDALONE EVALUATION STEP ENGINE
# =====================================================================
def evaluate_model(eval_model, loader):
    """
    Runs inference in strict evaluation mode (deactivates dropout layers)
    and computes financial variance metrics.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    eval_model = eval_model.to(device)
    eval_model.eval()
    
    all_preds = []
    all_targets = []
    
    # Disable gradient tracking to accelerate inference speed
    with torch.no_grad():
        for batch in loader:
            details, tabular, price, targets = batch
            
            # Send variables to active runtime device
            details = details.long().to(device)
            tabular = tabular.float().to(device) # Moved tabular branch data to device
            price = price.float().to(device)
            
            # Forward Pass through your multi-branch model
            outputs = eval_model(details, tabular, price)
            
            all_preds.extend(outputs.view(-1).cpu().tolist())
            all_targets.extend(targets.view(-1).cpu().tolist())
            
    # Convert lists to NumPy arrays for vectorized asset calculations
    p_real = np.asarray(all_preds)
    t_real = np.asarray(all_targets)
    
    mae = mean_absolute_error(t_real, p_real)
    r2 = r2_score(t_real, p_real)
    
    # Absolute Tolerance Check: lands within 0.1 standard deviation boundaries
    within_tol = np.abs(p_real - t_real) <= 0.1
    tol_acc = np.mean(within_tol)
    
    return mae, r2, p_real, t_real, tol_acc

# =====================================================================
# 3. RUN EVALUATION INFERENCE
# =====================================================================
print(f"🚀 Iniciando processamento da sessão de avaliação real ({len(val_ds)} ações)...\n")

# Leverage your trained model object directly
val_mae, val_r2, val_preds, val_targets, tol_acc = evaluate_model(model, val_loader)

# Print telemetry summaries to log performance
print("=" * 65)
print("--- [EVALUATION SESSION PERFORMANCE ANALYSIS] ---")
print("=" * 65)
print(f"Validation Mean Absolute Error (MAE) : {val_mae:.4f} Z-units")
print(f"Validation Coefficient of Fit (R²)   : {val_r2:.4f}")
print(f"Precisão dentro de 0.1 Desvios Padrão: {tol_acc:.2%}")

# =====================================================================
# 4. FINANCIAL DIRECTIONAL ACCURACY RADAR
# =====================================================================
# Symmetrical binary comparison handles zero-boundary conditions securely
directional_match = (val_preds >= 0) == (val_targets >= 0)
dir_acc = np.mean(directional_match)

print(f"Precisão Direcional do Mercado (Sign): {dir_acc:.2%}")
print("=" * 65)
print("\n✅ Sessão de avaliação finalizada. O modelo foi testado com sucesso.")

In [ ]:
#### [AI-CONTEXT]
#### ID: 027
#### ROLE: Automated Trading Simulator Block.
#### INPUT: Complete evaluation process (successful execution of ID: 026).
#### OUTPUT: Simulate a long/short portfolio strategy.

#### Estrategy: It assumes the user buys stocks with positive predictions and shorts the stocks with negative predictions on Day 61, using the val_preds and val_targets generated by the evaluation session.


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =====================================================================
# AUTOMATED TRADING SIMULATION (STRICT SEQUENTIAL PARITY)
# =====================================================================

# 1. FIXED: Extract true nominal lookback stats directly from the matching evaluation slice
# Dynamically extract the exact length of the evaluation predictions to prevent hardcoded size index bugs
num_eval_samples = len(val_preds)
eval_df_slice = df.iloc[-num_eval_samples:].copy()

# Isolate historical price columns to re-calculate pristine unscaled lookback parameters
price_cols_meta = [c for c in df.columns if str(c).replace('-', '').isdigit()]
historical_cols_meta = [c for c in price_cols_meta if str(c) != '0']

# Re-extract local row parameters directly via Pandas to guarantee perfect row-by-row alignment
row_means = eval_df_slice[historical_cols_meta].mean(axis=1).values
row_stds = eval_df_slice[historical_cols_meta].std(axis=1, ddof=0).values
eps = 1e-6

# 2. Reconstruct the true absolute nominal prices for Day 61 (Targets)
# Formula: Price = (Z_Score * Local_Row_Std) + Local_Row_Mean
day_61_real_prices = (val_targets * (row_stds + eps)) + row_means

# 3. Extract true absolute nominal prices for Day 60 directly from your master DataFrame
# This corresponds to column '0' (Today's Price) before any transformations or clips!
day_60_real_prices = eval_df_slice['0'].values if '0' in eval_df_slice.columns else eval_df_slice.iloc[:, -1].values

# 4. Reconstruct the model's predicted absolute nominal prices for Day 61
day_61_pred_prices = (val_preds * (row_stds + eps)) + row_means

# 5. Compute the true financial percentage return of Day 61 relative to Day 60
real_returns_pct = ((day_61_real_prices - day_60_real_prices) / (day_60_real_prices + 1e-8)) * 100.0

# 6. Determine execution signals based on predictive models
# FIXED: Updated strategy signals to strictly match your long/short strategy statement requirements:
# "buys stocks with positive predictions and shorts the stocks with negative predictions on Day 61"
trading_signals = np.where(val_preds >= 0, 1.0, -1.0)

# 7. Compute strategy performance per asset bucket
strategy_asset_returns = trading_signals * real_returns_pct

long_mask = trading_signals == 1
short_mask = trading_signals == -1

# 8. Aggregate returns assuming simultaneous equal-weighted multi-asset allocation
avg_long_return = np.mean(real_returns_pct[long_mask]) if np.any(long_mask) else 0.0
avg_short_return = np.mean(-real_returns_pct[short_mask]) if np.any(short_mask) else 0.0
portfolio_total_return = np.mean(strategy_asset_returns)

# =====================================================================
# PERFORMANCE DISPLAY LOGS
# =====================================================================
print("=" * 75)
print("--- [PORTFOLIO SIMULATION METRICS - STRICT PORTFOLIO PARITY] ---")
print("=" * 75)
print(f"Total Traded Assets in Portfolio : {len(val_targets)}")
print(f"Number of Long Positions         : {np.sum(long_mask)}")
print(f"Number of Short Positions        : {np.sum(short_mask)}")
print("-" * 75)
print(f"Average Yield on Long Legs       : {avg_long_return:.2f}%")
print(f"Average Yield on Short Legs      : {avg_short_return:.2f}%")
print(f"TOTAL SIMULTANEOUS PORTFOLIO NET RETURN: {portfolio_total_return:.2f}%")
print("=" * 75)

# --- VISUAL DISTRIBUTION OF REAL PERFORMANCE ---
plt.figure(figsize=(12, 5))
plt.hist(strategy_asset_returns, bins=25, color='teal', edgecolor='black', alpha=0.7)
plt.axvline(portfolio_total_return, color='darkred', linestyle='-', linewidth=2.5, 
            label=f'Portfolio Mean Return ({portfolio_total_return:.2f}%)')
plt.axvline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Distribution of Real Percentage Returns across Strategy Positions (Sequential Mode)', fontsize=12, fontweight='bold')
plt.xlabel('Individual Asset Strategy Return (%)')
plt.ylabel('Number of Stocks')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#### [AI-CONTEXT]
#### ID: 028
#### ROLE: Weight Exporter Script.
#### INPUT: Complete evaluation process (successful execution of ID: 026).
#### OUTPUT: Safely extracts the weights from the active GPU/Jupyter workspace, detaches them from the execution graphs, and saves them to the local directory.


import os
import torch

# =====================================================================
# PRODUCTION WEIGHT EXPORTER SCRIPT
# =====================================================================
export_dir = "/home/andre/AI_Lab/20260623_Project_StockMarket_App/production_models"
os.makedirs(export_dir, exist_ok=True)

export_path = os.path.join(export_dir, "stock_predictor_90pc_accuracy.pt")

# Save state dictionary (best practice for deploying architectures)
try:
    torch.save(model.state_dict(), export_path)
    print("=" * 65)
    print("✅ Model weights compiled and exported successfully!")
    print(f"Local Storage Destination: {export_path}")
    print("=" * 65)
except Exception as e:
    print(f"❌ Critical Export Failure: {str(e)}")

In [ ]:
#### [AI-CONTEXT]
#### ID: 029
#### ROLE: Attention Feature Importance Map.
#### INPUT: Complete evaluation process (successful execution of ID: 026).
#### OUTPUT: Extract the raw weights from the feature_attention gating network inside the trained model. It evaluates how the model prioritizes your primary price anchor relative to the 14 macroeconomic indicators across the dataset.


import numpy as np
import torch
import matplotlib.pyplot as plt

# =====================================================================
# ATTENTION FEATURE IMPORTANCE RADAR (PRICE TIMELINE REALIGNMENT)
# =====================================================================
# Reconstruct the explicit indicator sequence manually since 'tensors' was dropped from RAM
yahoo_keys = ["GOLD", "USD_BRL", "IBOV", "US2YT", "EWZ", "IBX50", "US500", "BTC_USD", "IMAB", "WTI", "VIX", "IFIX"]
bcb_keys = ["CDI", "IPCA"]
indicator_names = ["STOCK_PRICE"] + yahoo_keys + bcb_keys

# Place model in evaluation mode
model.eval()
with torch.no_grad():
    # Use the true 3D evaluation matrix (batch, 60, 15) to pull valid weights from memory
    # FIXED: Ensure model is mapped to the current device, tracking fallback correctly
    current_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(current_device)
    sample_input = master_input_eval_3d_tensor.float().to(current_device)
    
    # Extract and flatten features exactly as performed in forward()
    batch_size, seq_len, num_features = sample_input.shape
    x_flat = sample_input.contiguous().view(-1, num_features)
    
    # Query the gating layers directly using the corrected 15-feature matrix
    attention_weights = model.price_branch.feature_attention(x_flat)
    
    # Reshape back and compute the mean activation weight for each of the 15 parameters across stocks and days
    mean_weights = attention_weights.view(batch_size, seq_len, num_features).mean(dim=[0, 1]).cpu().numpy()

# Normalize weights to sum up to 100% for clear relative plotting
normalized_importance = (mean_weights / np.sum(mean_weights)) * 100

# Sort items in descending order for rapid scannability
sorted_indices = np.argsort(normalized_importance)[::-1]
sorted_names = [indicator_names[i] for i in sorted_indices]
sorted_importance = normalized_importance[sorted_indices]

# =====================================================================
# RENDERING PLOT CHART
# =====================================================================
plt.figure(figsize=(12, 6))

# Evaluate directly against the string name to maintain robust color alignment after sorting
colors = ['darkblue' if name == "STOCK_PRICE" else 'skyblue' for name in sorted_names]
bars = plt.bar(sorted_names, sorted_importance, color=colors, edgecolor='black', alpha=0.8)

plt.title('Macro Benchmark Prioritization Map (Dynamic Attention Gates)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Relative Feature Importance Score (%)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Annotate exact percentage values over the visual bars
for bar in bars:
    height = bar.get_height()
    plt.annotate(f'{height:.1f}%',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3),  
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
#### [AI-CONTEXT]
#### ID: 030
#### ROLE: Real-world target tomorrow's prices with actual neural network forecasts.
#### INPUT: Complete evaluation process (successful execution of ID: 026).
#### OUTPUT: Calculate the target vectors, reverse the mathematical scale, and append them as the final three columns of your master dataset.


import torch
import numpy as np
import pandas as pd

# =====================================================================
# 1. INITIALIZE INFRASTRUCTURE & RE-ALIGN DATAFRAME FOOTPRINT
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Deploying low-memory batching forecasting engine on device: {device}")

if 'model' in globals():
    model.to(device)
    model.eval()  
else:
    raise NameError("❌ Deployment Error: The trained 'model' instance was not found in memory.")

# Re-align raw inputs
full_master_3d_tensor = torch.cat([master_input_train_3d_tensor, master_input_eval_3d_tensor], dim=0)
details_full_tensor = torch.cat([details_train_tensor, details_eval_tensor], dim=0)

# FIXED: Concatenate training and validation tabular features to keep parity with full_master_3d_tensor
tabular_full_tensor = torch.cat([liquidity_train_tensor, liquidity_eval_tensor], dim=0)

# Extract parameters directly via Pandas to guarantee perfect row-by-row alignment
price_cols_meta = [c for c in df.columns if str(c).replace('-', '').isdigit() or str(c) == '0']
historical_cols_meta = [c for c in price_cols_meta if str(c) != '0']

df_means = df[historical_cols_meta].mean(axis=1).values
df_stds = df[historical_cols_meta].std(axis=1, ddof=0).values

# Prepare empty placeholder column fields inside the main dataframe layout safely
df['TOMORROW_PRED_MLP'] = np.nan
df['DAY5_PRED_MLP'] = np.nan
df['DAY10_PRED_MLP'] = np.nan
df['PEAK_10D_PRED_MLP'] = np.nan  

# Dynamically look up column index positions to accelerate iat operations
col_0_pos = df.columns.get_loc('0')
col_tomorrow_pos = df.columns.get_loc('TOMORROW_PRED_MLP')
col_day5_pos = df.columns.get_loc('DAY5_PRED_MLP')
col_day10_pos = df.columns.get_loc('DAY10_PRED_MLP')
col_peak_pos = df.columns.get_loc('PEAK_10D_PRED_MLP')

# =====================================================================
# 2. MINI-BATCHED AUTOREGRESSIVE INFERENCE ENGINE (RAM SAFE)
# =====================================================================
batch_size = 64  
total_assets = full_master_3d_tensor.shape[0]

with torch.no_grad():
    for start_idx in range(0, total_assets, batch_size):
        end_idx = min(start_idx + batch_size, total_assets)
        
        batch_features = full_master_3d_tensor[start_idx:end_idx].float().to(device)
        batch_details = details_full_tensor[start_idx:end_idx].long().to(device)
        batch_tabular = tabular_full_tensor[start_idx:end_idx].float().to(device) # Moved tabular tensor to device
        
        batch_future_preds = []
        
        # Run 10-day rolling loop step safely
        for step in range(10):
            # FIXED: Passed batch_tabular to satisfy the triple-branch forward layout method signature
            outputs = model(batch_details, batch_tabular, batch_features)
            
            # CRASH-PROOF CONSTRAINT: Prevent Z-score explosion by capping recursive loops
            outputs_bounded = torch.clamp(outputs, min=-3.5, max=3.5)
            batch_future_preds.append(outputs_bounded.clone())
            
            # Roll time-series window sequence matrix indices forward
            new_rolling = batch_features[:, 1:, :].clone()
            
            next_day_vector = torch.zeros((batch_features.shape[0], 1, 15), device=device)
            next_day_vector[:, 0, 0] = outputs_bounded.view(-1)
            next_day_vector[:, 0, 1:] = batch_features[:, -1, 1:].clone()
            batch_features = torch.cat((new_rolling, next_day_vector), dim=1)
            
        # Reconstruct output matrix configurations (batch, 10)
        standardized_forecast_matrix = torch.cat(batch_future_preds, dim=1).cpu().numpy()
        
        # =====================================================================
        # 3. CRASH-PROOF POSITION-BASED DE-STANDARDIZATION
        # =====================================================================
        for local_idx in range(standardized_forecast_matrix.shape[0]):
            global_idx = start_idx + local_idx
            
            # Bounds checking protects against tensor array changes
            if global_idx >= len(df):
                continue
                
            mu = df_means[global_idx]
            sigma = df_stds[global_idx] + 1e-6
            
            # Unscale the full 10-day projection track into raw prices
            raw_forecast_track = (standardized_forecast_matrix[local_idx, :] * sigma) + mu
            
            # PHYSICAL BOUNDARY FLOOR: A stock price can never legally drop below R$ 0.01
            raw_forecast_track = np.clip(raw_forecast_track, a_min=0.01, a_max=None)
            
            tomorrow_price = raw_forecast_track[0]
            day5_price = raw_forecast_track[4]
            day10_price = raw_forecast_track[9]
            
            # Use .iat with integer positions to bypass label mismatches
            today_entry_price = df.iat[global_idx, col_0_pos] if '0' in df.columns else tomorrow_price
            max_horizon_peak = max(np.max(raw_forecast_track), today_entry_price * 1.01)
            
            # Direct placement via integer coordinates eliminates index errors
            df.iat[global_idx, col_tomorrow_pos] = tomorrow_price
            df.iat[global_idx, col_day5_pos] = day5_price
            df.iat[global_idx, col_day10_pos] = day10_price
            df.iat[global_idx, col_peak_pos] = max_horizon_peak

# =====================================================================
# 4. RENDER VERIFIED PORTFOLIO SCREENER
# =====================================================================
columns_to_show = ['COMPANY', 'TOMORROW_PRED_MLP', 'DAY5_PRED_MLP', 'PEAK_10D_PRED_MLP']
print("=" * 115)
print(f"🎯 [CRASH-PROOF FORECAST COMPLETE - DATA ARCHITECTURE STABILIZED: {df.shape}]")
print("=" * 115)
print(df[columns_to_show].dropna().head(10).to_string(index=False, formatters={
    'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
    'DAY5_PRED_MLP': 'R$ {:,.2f}'.format,
    'PEAK_10D_PRED_MLP': 'R$ {:,.2f}'.format
}))
print("=" * 115)

In [ ]:
#### [AI-CONTEXT]
#### ID: 031
#### ROLE: Alpha Generator.
#### INPUT: Complete generation of Real-world target and autoregressive price forecast (successful execution of ID: 030).
#### OUTPUT: Calculate the percentage returns relative to tomorrow's baseline price, sort the portfolio, and display the two extreme 5-stock baskets.


import pandas as pd
import numpy as np

# =====================================================================
# 1. CALCULATE EXPECTED PERCENTAGE RETURNS USING PEAK HORIZONS
# =====================================================================
df['EXPECTED_5D_RETURN_PCT'] = ((df['PEAK_10D_PRED_MLP'] - df['TOMORROW_PRED_MLP']) / df['TOMORROW_PRED_MLP']) * 100
df['SHORT_POTENTIAL_PCT'] = ((df['DAY10_PRED_MLP'] - df['TOMORROW_PRED_MLP']) / df['TOMORROW_PRED_MLP']) * 100

# =====================================================================
# 2. SYMMETRICAL CRASH-PROOF VOLATILITY TRUNCATION LAYER
# =====================================================================
# This section acts as a symmetrical risk filter that screens out high-risk anomalies and ensures the trading strategies only pick realistic, actionable assets.
# Isolate valid, bounded pools for long and short allocations
long_mask = (df['EXPECTED_5D_RETURN_PCT'] > 0.1) & (df['EXPECTED_5D_RETURN_PCT'] <= 25.0)
df_long_pool = df[long_mask].copy()

# Short mask captures valid downward trends within a realistic 25% systemic boundary
short_mask = (df['SHORT_POTENTIAL_PCT'] < -0.1) & (df['SHORT_POTENTIAL_PCT'] >= -25.0)
df_short_pool = df[short_mask].copy()

# =====================================================================
# 3. SEPARATE, SORT, AND EXTRACT ARCHITECTURAL ALPHA BASKETS
# =====================================================================
long_sorted = df_long_pool.sort_values(by='EXPECTED_5D_RETURN_PCT', ascending=False)
short_sorted = df_short_pool.sort_values(by='SHORT_POTENTIAL_PCT', ascending=True)

# FIXED: Replaced 'INDEX' with df.index.name or fallback identifier string
# If 'INDEX' is not present as a literal column name, using df.index or df.columns[0] prevents a KeyError lookup failure.
col_0_name = df.columns[0] if 'INDEX' not in df.columns else 'INDEX'

columns_layout = [col_0_name, 'COMPANY', 'SECTOR', 'TOMORROW_PRED_MLP', 'PEAK_10D_PRED_MLP', 'EXPECTED_5D_RETURN_PCT']
short_layout = [col_0_name, 'COMPANY', 'SECTOR', 'TOMORROW_PRED_MLP', 'DAY10_PRED_MLP', 'SHORT_POTENTIAL_PCT']

top_5_outperformed = long_sorted.head(5)[columns_layout]
top_5_underperformed = short_sorted.head(5)[short_layout]

# =====================================================================
# 4. DISPLAY PERFORMANCE RADAR TABLES
# =====================================================================
print("=" * 115)
print(f"🚀 [TOP 5 OUTPERFORMED STOCKS - LONG CANDIDATES - FILTERED POOL SIZE: {len(df_long_pool)}]")
print("=" * 115)
if not top_5_outperformed.empty:
    print(top_5_outperformed.to_string(index=False, formatters={
        'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
        'PEAK_10D_PRED_MLP': 'R$ {:,.2f}'.format,
        'EXPECTED_5D_RETURN_PCT': '{:,.2f}%'.format
    }))
else:
    print("⚠️ No long positions survived the structural 25% volatility filter.")

print("\n" + "=" * 115)
print(f"🛑 [TOP 5 UNDERPERFORMED STOCKS - SHORT CANDIDATES - BOUNDED POOL SIZE: {len(df_short_pool)}]")
print("=" * 115)
if not top_5_underperformed.empty:
    print(top_5_underperformed.to_string(index=False, formatters={
        'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
        'DAY10_PRED_MLP': 'R$ {:,.2f}'.format,
        'SHORT_POTENTIAL_PCT': '{:,.2f}%'.format
    }))
else:
    print("⚠️ No short positions detected within the valid -25% systemic boundary.")
print("=" * 115)

In [ ]:
#### [AI-CONTEXT]
#### ID: 032
#### ROLE: RSI calculator.
#### INPUT: Complete generation of Real-world target and autoregressive price forecast (successful execution of ID: 030).
#### OUTPUT: Calculate the Relative Strength Index (RSI) for each stock across its final 14 pricing steps.


import torch
import numpy as np
import pandas as pd

# =====================================================================
# 1. ANTI-SHIFT STRUCTURAL HANDSHAKE (DYNAMIC ALIGNMENT)
# =====================================================================
if df['COMPANY'].isna().any():
    df = df.dropna(subset=['COMPANY']).copy()
    df = df.reset_index(drop=True)

print(f"🧹 Pipeline Aligned! Active DataFrame Screener Size: {df.shape}")

# =====================================================================
# 2. INITIALIZE HARDWARE ENVIRONMENT AND ARRAYS
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Deploying global RSI computational engine on hardware device: {device}")

# Recombine the raw unscaled structural sequence rows from our splits
# FIXED: Using unstandardized price arrays (price_train_tensor and price_eval_tensor from ID 009)
# calculating RSI from standardized parameters distorts price change ratios.
raw_unscaled_prices = torch.cat([price_train_tensor, price_eval_tensor], dim=0).to(device)
real_prices_15d = raw_unscaled_prices[:, -15:].to(torch.float32)

# =====================================================================
# 3. COMPUTE WILDER'S SMOOTHED MOMENTUM ALTERNATIVES (CANONICAL RSI)
# =====================================================================
price_changes = real_prices_15d[:, 1:] - real_prices_15d[:, :-1]
gains = torch.clamp(price_changes, min=0.0)
losses = torch.clamp(-price_changes, min=0.0)

# Initialize Wilder's exponential smoothing accumulation vectors
num_assets = gains.shape[0]
prev_avg_gain = torch.mean(gains[:, :1], dim=1, keepdim=True)
prev_avg_loss = torch.mean(losses[:, :1], dim=1, keepdim=True)

# Smooth vectors iteratively across the remaining 13 time intervals
for t in range(1, 14):
    prev_avg_gain = (prev_avg_gain * 13.0 + gains[:, t:t+1]) / 14.0
    prev_avg_loss = (prev_avg_loss * 13.0 + losses[:, t:t+1]) / 14.0

eps = 1e-8
rs = prev_avg_gain / (prev_avg_loss + eps)
rsi_scores = 100.0 - (100.0 / (1.0 + rs))
rsi_flat_numpy = rsi_scores.detach().cpu().flatten().numpy()

# =====================================================================
# 4. CRASH-PROOF SECTORED LOOKUP MAPPING VIA HISTORICAL ALIGNMENT
# =====================================================================
# FIXED: Fallback vector maps directly to target_sample_size (1579) matching raw tensor dimensions 
if 'recombined_dataframe_indices' in globals():
    active_mapping_indices = recombined_dataframe_indices
else:
    active_mapping_indices = np.arange(num_assets)

df['RSI_14'] = np.nan

# Map arrays safely using the true un-truncated historical tensor row coordinate
for df_idx, row in df.iterrows():
    # FIXED: Check boundary against current active dataframe index safely
    if df_idx < len(active_mapping_indices) and df_idx < len(rsi_flat_numpy):
        tensor_row_idx = active_mapping_indices[df_idx]
        df.at[df_idx, 'RSI_14'] = rsi_flat_numpy[tensor_row_idx]

# =====================================================================
# 5. DYNAMIC SCREENER DISPLAY
# =====================================================================
columns_to_show = ['INDEX', 'COMPANY', 'SECTOR', 'TOMORROW_PRED_MLP', 'RSI_14']

print("=" * 110)
print("📈 [CURRENT 14-PERIOD RSI INVENTORY SUMMARY COMPLETE - CANONICAL WILDER ALIGNED]")
print("=" * 110)
print(df[columns_to_show].dropna(subset=['TOMORROW_PRED_MLP', 'RSI_14']).head(10).to_string(index=False, formatters={
    'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
    'RSI_14': '{:,.2f}'.format
}))
print("=" * 110)

In [ ]:
#### [AI-CONTEXT]
#### ID: 033
#### ROLE: 20 and 50 SMA calculator.
#### INPUT: Complete generation of Real-world target and autoregressive price forecast (successful execution of ID: 030).
#### OUTPUT: Calculate the 20-day Simple Moving Average (SMA) and 50-day Simple Moving Average (SMA) rolling over the last 5 days for each stock.


import torch
import numpy as np
import pandas as pd

# =====================================================================
# 1. ANTI-SHIFT STRUCTURAL HANDSHAKE (DYNAMIC ALIGNMENT)
# =====================================================================
if df['COMPANY'].isna().any():
    df = df.dropna(subset=['COMPANY']).copy()
    df = df.reset_index(drop=True)

print(f"🧹 Pipeline Aligned! Active DataFrame Screener Size: {df.shape}")

# =====================================================================
# 2. EXTRACT PRISTINE HISTORICAL DATA FROM RECOMBINED TRACKS
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Deploying global SMA computational engine on hardware device: {device}")

# Combine raw historical structural columns back into contiguous layout maps
raw_unscaled_prices = torch.cat([price_train_tensor, price_eval_tensor], dim=0).cpu().numpy()
num_total_assets = raw_unscaled_prices.shape[0]

# Isolate exactly the final 54 days of historical data from the 61-day grid
# FIXED: The first slice of a 50-day window shifting across 5 days requires index 0 to 49.
# This means we need 4 days of history before the final 5 days window start. 
# Total index range must cover 54 elements: 4 + 50 = 54 elements.
raw_prices_54d = raw_unscaled_prices[:, -54:]

# =====================================================================
# 3. CALCULATE ROLLING 20 SMA & 50 SMA FOR THE LAST 5 DAYS
# =====================================================================
sma20_days_list = []
sma50_days_list = []

for i in range(5):
    # FIXED: Re-aligned indexing loops to properly roll forward sequentially over time 
    # to calculate the moving metrics across each of the 5 targeted execution intervals.
    start_idx_20 = i + 30  # Slices indices 30 to 50 on step 0 to calculate the 20-period map
    end_idx_20 = start_idx_20 + 20
    sma20_step = np.mean(raw_prices_54d[:, start_idx_20:end_idx_20], axis=1)
    sma20_days_list.append(sma20_step)
    
    start_idx_50 = i
    end_idx_50 = start_idx_50 + 50
    sma50_step = np.mean(raw_prices_54d[:, start_idx_50:end_idx_50], axis=1)
    sma50_days_list.append(sma50_step)

sma20_matrix = np.column_stack(sma20_days_list)
sma50_matrix = np.column_stack(sma50_days_list)

# =====================================================================
# 4. CRASH-PROOF SYNCHRONIZED SMA DATAFRAME INTEGRATION
# =====================================================================
for day_offset in range(5):
    day_label = f"LATEST" if day_offset == 4 else f"{4 - day_offset}D_AGO"
    df[f'SMA20_{day_label}'] = np.nan
    df[f'SMA50_{day_label}'] = np.nan

# Establish alignment array reference maps securely
if 'recombined_dataframe_indices' in globals():
    active_mapping_indices = recombined_dataframe_indices
else:
    active_mapping_indices = np.arange(num_total_assets)

# Map matrix metrics using the true historical index position anchors
for df_idx, row in df.iterrows():
    if df_idx < len(active_mapping_indices):
        tensor_row_idx = active_mapping_indices[df_idx]
        if tensor_row_idx < num_total_assets: # Added index safety check
            for day_offset in range(5):
                day_label = f"LATEST" if day_offset == 4 else f"{4 - day_offset}D_AGO"
                df.at[df_idx, f'SMA20_{day_label}'] = sma20_matrix[tensor_row_idx, day_offset]
                df.at[df_idx, f'SMA50_{day_label}'] = sma50_matrix[tensor_row_idx, day_offset]

# =====================================================================
# 5. CALCULATE RELATIVE TREND SPREADS & BUILD MASTER 3D TENSOR
# =====================================================================
eps = 1e-8
relative_spread_matrix = (sma20_matrix - sma50_matrix) / (sma20_matrix + eps)

# Cast NumPy matrix into a 3D PyTorch tensor -> Shape: (batch_size, 5, 1)
spread_tensor_2d = torch.tensor(relative_spread_matrix, dtype=torch.float32)
sma_spread_master_tensor = spread_tensor_2d.unsqueeze(-1)

# =====================================================================
# VERIFICATION AND ROW QUERY SAMPLES
# =====================================================================
print("=" * 85)
print("🎯 [SMA SPREAD TRAJECTORY TENSOR MAP COMPLETE]")
print("=" * 85)
print(f"Master Spread Tensor Shape: {list(sma_spread_master_tensor.shape)} (Expected: [batch_size, 5, 1])")
print(f"Primary Dataframe Shape    : {df.shape}")
print("-" * 85)

if len(df) > 0 and len(active_mapping_indices) > 0:
    first_row_ticker = df.at[0, 'INDEX']
    first_tensor_row = active_mapping_indices[0]
    if first_tensor_row < len(sma_spread_master_tensor):
        stock_1x5_vector = sma_spread_master_tensor[first_tensor_row].view(1, 5)
        print(f"📋 Row Vector Check for Asset Code: {first_row_ticker}")
        print(f"Tensor Row Index: {first_tensor_row} | Shape: {list(stock_1x5_vector.shape)}")
        print(f"Contents:\n{stock_1x5_vector}")
print("=" * 85)

In [ ]:
#### [AI-CONTEXT]
#### ID: 034
#### ROLE: Momentum with the linear regression slope.
#### INPUT: Complete 20 SMA and 50 SMA calculation (successful execution of ID: 033).
#### OUTPUT: Calculate the linear regression slope beta_1 and intercept beta_0 for each stock's 5-day SMA relative spread trajectory using parallelized vector mathematics. The 5 sequential coordinates were treated as values along a standardized geometric timeline: (X = [0.0, 1.0, 2.0, 3.0, 4.0]).

import torch
import numpy as np
import pandas as pd

# =====================================================================
# 1. ANTI-SHIFT STRUCTURAL HANDSHAKE (DYNAMIC ALIGNMENT)
# =====================================================================
if df['COMPANY'].isna().any():
    df = df.dropna(subset=['COMPANY']).copy()
    df = df.reset_index(drop=True)

print(f"🧹 Pipeline Aligned! Active DataFrame Screener Size: {df.shape}")

# =====================================================================
# 2. INITIALIZE HARDWARE ENVIRONMENT AND STANDARDIZED ARRAYS
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Deploying Balanced Geometric Polar Engine on hardware device: {device}")

y_values = sma_spread_master_tensor.squeeze(-1).to(device)

# Programmatically track the clean integer row count from axis 0
num_total_assets = y_values.shape[0]

# Unit-integer timeline represents standard daily steps
x_values = torch.tensor([0.0, 1.0, 2.0, 3.0, 4.0], dtype=torch.float32, device=device)
n_points = 5.0

sum_x = torch.sum(x_values)                             # 10.0
sum_x_sq = torch.sum(x_values ** 2)                     # 30.0
ss_xx = sum_x_sq - ((sum_x ** 2) / n_points)            # Clean scale denominator (10.0)

# =====================================================================
# 3. RUN VECTORIZED OLS REGRESSION (ON-DEVICE SPEED)
# =====================================================================
sum_y = torch.sum(y_values, dim=1)
sum_xy = torch.sum(y_values * x_values, dim=1)

ss_xy = sum_xy - ((sum_x * sum_y) / n_points)
slopes_raw = ss_xy / ss_xx

# Scale raw slope to percentage space (basis points per day) 
# to balance the vertical vector against the 1.0 horizontal time unit
slopes = slopes_raw * 100.0

# Isolate position vectors to capture true long-term baseline territories
current_spread_position = y_values[:, 4]
polar_position_signs = torch.sign(current_spread_position)

# =====================================================================
# 4. VECTORIZED POLAR CONVERSION (BALANCED PERCENTAGE SCALE MATRIX)
# =====================================================================
# Synchronized Cartesian horizontal delta step matching our 1-day time unit
cartesian_x = torch.tensor(1.0, device=device)

# Magnitude reflects absolute baseline depth signed by position
raw_magnitude = torch.sqrt((cartesian_x ** 2) + (slopes ** 2))
polar_magnitude = raw_magnitude * polar_position_signs

# Compute custom angular headings in radians and convert to degrees
polar_angle_rad = torch.atan2(slopes, cartesian_x)
polar_angle_deg = polar_angle_rad * (180.0 / np.pi)

mag_flat_numpy = polar_magnitude.cpu().numpy()
ang_flat_numpy = polar_angle_deg.cpu().numpy()

# =====================================================================
# 5. CRASH-PROOF ALIGNMENT MAPPING VIA HISTORICAL TRACKS
# =====================================================================
df['POLAR_MAGNITUDE'] = np.nan
df['POLAR_ANGLE_DEG'] = np.nan

if 'recombined_dataframe_indices' in globals():
    active_mapping_indices = recombined_dataframe_indices
else:
    active_mapping_indices = np.arange(num_total_assets)

for df_idx, row in df.iterrows():
    if df_idx < len(active_mapping_indices):
        tensor_row_idx = active_mapping_indices[df_idx]
        if tensor_row_idx < len(mag_flat_numpy):  # Added bounds check to guarantee zero runtime failures
            df.at[df_idx, 'POLAR_MAGNITUDE'] = mag_flat_numpy[tensor_row_idx]
            df.at[df_idx, 'POLAR_ANGLE_DEG'] = ang_flat_numpy[tensor_row_idx]

# =====================================================================
# 6. DYNAMIC SCREENER DISPLAY
# =====================================================================
columns_to_show = ['INDEX', 'SECTOR', 'POLAR_MAGNITUDE', 'POLAR_ANGLE_DEG']

print("=" * 110)
print("🌐 [GPU HARDWARE ACCELERATED GEOMETRIC POLAR COEFFICIENTS COMPLETE - PERCENTAGE SCALE LOCKED]")
print("=" * 110)
print(df[columns_to_show].dropna(subset=['POLAR_MAGNITUDE']).head(10).to_string(index=False, formatters={
    'POLAR_MAGNITUDE': '{:,.4f}'.format,
    'POLAR_ANGLE_DEG': '{:,.2f}°'.format
}))
print("=" * 110)

In [ ]:
#### [AI-CONTEXT]
#### ID: 035
#### ROLE: Calculate the option with better buy-in prices.
#### INPUT: Complete calculation of the Momentum with the linear regression slope. (successful execution of ID: 033).
#### OUTPUT: dataframe buy_in_table.


import pandas as pd
import numpy as np
import feedparser
import urllib.parse
import re
from datetime import datetime, timezone, timedelta

# Create a copy to protect original workspace layout maps
df_pipeline = df.copy()

# =====================================================================
# STEP 1 (RULE 1): Group the 200 highest values of EXPECTED_5D_RETURN_PCT
# =====================================================================
df_pipeline = df_pipeline.nlargest(200, 'EXPECTED_5D_RETURN_PCT')

# =====================================================================
# STEP 2 (RULE 2): Purge and keep only RSI_14 values below 35
# =====================================================================
df_pipeline = df_pipeline[df_pipeline['RSI_14'] < 35]

# =====================================================================
# STEP 3 & 4 (RULES 3 & 4): RESTORED STRICT USER PREFERENCE FILTERS
# =====================================================================
# Targets assets deep below their long-term trend lines (negative magnitude)
# that are simultaneously demonstrating a strong short-term upward reversal (positive angle)
df_pipeline = df_pipeline[df_pipeline['POLAR_MAGNITUDE'] < 0]
df_pipeline = df_pipeline[df_pipeline['POLAR_ANGLE_DEG'] > 0]

# =====================================================================
# STEPS 5 to 7 (RULES 5, 6, 7): Multi-tiered Final Ordering
# =====================================================================
buy_in_table = df_pipeline.sort_values(
    by=['RSI_14', 'POLAR_MAGNITUDE', 'POLAR_ANGLE_DEG'],
    ascending=[False, True, False]
)

# =====================================================================
# STEP 8 (RULE 8): Select the top 10 rows and filter required columns
# =====================================================================
target_columns = [
    'INDEX', 'COMPANY', 'SECTOR', 'SUBSECTOR', '0', 'TOMORROW_PRED_MLP', 
    'DAY5_PRED_MLP', 'EXPECTED_5D_RETURN_PCT', 'RSI_14', 
    'POLAR_MAGNITUDE', 'POLAR_ANGLE_DEG'
]

available_cols = [c for c in target_columns if c in buy_in_table.columns]
buy_in_table = buy_in_table[available_cols].head(10).reset_index(drop=True)
buy_in_table = buy_in_table.rename(columns={"0": "TODAY_PRICE"})

# =====================================================================
# STEP 9: Integrated Real-Time Google News Volume (Dual Ticker-Fallback)
# =====================================================================
def get_recent_news_volume(keyword):
    if pd.isna(keyword) or not str(keyword).strip():
        return 0
        
    encoded_keyword = urllib.parse.quote(str(keyword).strip())
    rss_url = f"https://news.google.com/rss/search?q={encoded_keyword}&hl=pt-BR&gl=BR&ceid=BR:pt-419"
    
    try:
        feed = feedparser.parse(rss_url, agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64)')
        if not feed.entries:
            return 0
            
        now_utc = datetime.now(timezone.utc)
        cutoff_time = now_utc - timedelta(hours=48)
        
        recent_count = 0
        for entry in feed.entries:
            pub_date_str = entry.get('published')
            if pub_date_str:
                try:
                    pub_date = pd.to_datetime(pub_date_str, utc=True).to_pydatetime()
                    if pub_date >= cutoff_time:
                        recent_count += 1
                except Exception:
                    continue
        return recent_count
    except Exception:
        return 0

def classify_volume(count):
    if count == 0:
        return 'NO NEWS'
    elif 1 <= count <= 2:
        return 'GREEN'
    elif 3 <= count <= 6:
        return 'YELLOW'
    else:
        return 'RED'

if not buy_in_table.empty:
    print("🔄 Pulling real-time Google News trends for deep-value turnaround candidates...")
    volumes_int = []
    
    for index, row in buy_in_table.iterrows():
        ticker_symbol = str(row['INDEX']).strip()
        raw_company_name = str(row['COMPANY']).strip()
        
        # FIXED: Removed the invalid backslash escape syntax on 'S\.?A\.?' and similar tokens
        # inside the raw regex pattern. Wrapped with a clean raw string r'' to ensure 
        # the pattern compiles error-free.
        clean_keyword = pd.Series([raw_company_name]).str.replace(
            r'\b(S\.?A\.?|PART\.?|INC\.?|AG\.?|CORP\.?|LTDA\.?|INT\.?|THERA\.?|THERAPEUTICS|EDUC|CRI)\b', 
            '', regex=True, flags=re.IGNORECASE
        ).str.replace(r'[^\w\s-]', '', regex=True).str.strip().values[0]
        
        search_query = clean_keyword if clean_keyword else raw_company_name
        news_count = get_recent_news_volume(search_query)
        
        if news_count == 0 and ticker_symbol:
            news_count = get_recent_news_volume(ticker_symbol)
            
        volumes_int.append(news_count)
        
    buy_in_table['VOLUME_RECENT_NEWS'] = [classify_volume(c) for c in volumes_int]

# =====================================================================
# DISPLAY RESULTS
# =====================================================================
print("=" * 140)
print(f"🌟 [FINAL STRATEGIC BUY_IN_TABLE - {len(buy_in_table)} ASSETS GENERATED]")
print("=" * 140)
if not buy_in_table.empty:
    formatter_dict = {
        'TODAY_PRICE': 'R$ {:,.2f}'.format,
        'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
        'PEAK_10D_PRED_MLP': 'R$ {:,.2f}'.format,
        'EXPECTED_5D_RETURN_PCT': '{:,.2f}%'.format,
        'RSI_14': '{:,.2f}'.format,
        'POLAR_MAGNITUDE': '{:,.4f}'.format,
        'POLAR_ANGLE_DEG': '{:,.2f}°'.format,
        'VOLUME_RECENT_NEWS': '{}'.format
    }
    active_formatters = {k: v for k, v in formatter_dict.items() if k in buy_in_table.columns}
    print(buy_in_table.to_string(formatters=active_formatters))
else:
    print("⚠️ Warning: No assets survived the progressive purges (Rules 2, 3, or 4).")
print("=" * 140)

In [ ]:
buy_in_table.head(10)

In [ ]:
#### [AI-CONTEXT]
#### ID: 036
#### ROLE: Identify the stocks with the top sell-out prices.
#### INPUT: Complete calculation of the Momentum with the linear regression slope. (successful execution of ID: 033).
#### OUTPUT: dataframe sell_out_table.


import re
import pandas as pd
import numpy as np
import feedparser
import urllib.parse
from datetime import datetime, timezone, timedelta

# =====================================================================
# DEFINE RE-FIT STAGES (From High-Conviction Strict to Broad Fallbacks)
# =====================================================================
relaxation_stages = [
    {
        "stage": 1,
        "desc": "Strict Initial Rules (RSI > 68 & 2-day Consecutive Historical Rally)",
        "rsi_min": 68,
        "strict_trend": True
    },
    {
        "stage": 2,
        "desc": "Relaxed Trend Rule (RSI > 65 & At least 1-day Positive History)",
        "rsi_min": 65,
        "strict_trend": False
    },
    {
        "stage": 3,
        "desc": "Capitulation Fallback (RSI > 60 & Tomorrow Downward Drop Only)",
        "rsi_min": 60,
        "strict_trend": "tomorrow_only"
    }
]

pipeline_success = False
sell_out_table = pd.DataFrame()

print("=" * 125)
print("⚙️  [STARTING AUTOMATED POSITION MATCHING RADAR]")
print("=" * 125)

# FIXED: Dynamically capture exact column label mappings by location to eliminate KeyErrors
col_0_name = '0' if '0' in df.columns else df.columns[-4]

# Find the exact column labels that precede '0' in the dataframe structure
price_indices = [df.columns.get_loc(c) for c in df.columns if str(c).replace('-', '').isdigit() or str(c) == '0']
sorted_price_indices = sorted(price_indices)
loc_0 = df.columns.get_loc(col_0_name)

# Extract column names by precise historical offset arrays
col_minus1_name = df.columns[sorted_price_indices[sorted_price_indices.index(loc_0) - 1]]
col_minus2_name = df.columns[sorted_price_indices[sorted_price_indices.index(loc_0) - 2]]

for config in relaxation_stages:
    print(f"🔄 Attempting Stage {config['stage']}: {config['desc']}...")
    
    # Initialize a fresh copy of the master dataset layout for this attempt
    df_attempt = df.copy()
    
    # --- Execute Rule 1 (Trend Subtractions Based on Position Lookups) ---
    if config['strict_trend'] == True:
        # Asset peaks out: climbs for two days, then turns downward tomorrow
        trend_mask = (
            ((df_attempt['TOMORROW_PRED_MLP'] - df_attempt[col_0_name]) < 0) &
            ((df_attempt[col_0_name] - df_attempt[col_minus1_name]) > 0) &
            ((df_attempt[col_minus1_name] - df_attempt[col_minus2_name]) > 0)
        )
    elif config['strict_trend'] == False:
        # Relaxed history: Must turn down tomorrow, requiring today to be higher than yesterday
        trend_mask = (
            ((df_attempt['TOMORROW_PRED_MLP'] - df_attempt[col_0_name]) < 0) &
            ((df_attempt[col_0_name] - df_attempt[col_minus1_name]) > 0)
        )
    else:
        # Minimum baseline fallback: Only requires a negative prediction step tomorrow
        trend_mask = ((df_attempt['TOMORROW_PRED_MLP'] - df_attempt[col_0_name]) < 0)
        
    df_attempt = df_attempt[trend_mask]
    
    # --- Execute Rule 2 (Dynamic RSI Minimum Boundary) ---
    df_attempt = df_attempt[df_attempt['RSI_14'] > config['rsi_min']]
    
    # --- Execute Rule 3 & 4 (Polar Trend Coordination Parameters) ---
    # FIXED: Reversing rules to target an overextended trend peaking out and turning downward
    # Overbought trends have a positive structural depth (magnitude > 0) and lookback deceleration (angle < 0)
    df_attempt = df_attempt[df_attempt['POLAR_MAGNITUDE'] > 0]
    df_attempt = df_attempt[df_attempt['POLAR_ANGLE_DEG'] < 0]
    
    # Check if any asset survived this specific configuration stage
    if not df_attempt.empty:
        print(f"✅ Target isolated! {len(df_attempt)} assets matched the parameters of Stage {config['stage']}.\n")
        
        # --- Execute Rules 5, 6, 7 (Multi-tiered Final Ordering) ---
        # FIXED: Sort descending for overbought RSI indicators, but ascending for descending angles
        sell_out_table = df_attempt.sort_values(
            by=['RSI_14', 'POLAR_MAGNITUDE', 'POLAR_ANGLE_DEG'],
            ascending=[False, False, True]
        )
        
        pipeline_success = True
        break  
    else:
        print(f"❌ Stage {config['stage']} returned 0 rows. Dropping constraints down to the next fallback level...")

# =====================================================================
# DATA CLEANUP AND STRUCTURAL RENDERING
# =====================================================================
if pipeline_success:
    # Rename column '0' to 'TODAY_PRICE' directly in the working layout
    sell_out_table = sell_out_table.rename(columns={col_0_name: 'TODAY_PRICE'})
    
    # RESTORED: Matched your strict baseline 5-day horizon inventory setup
    target_columns = [
        'INDEX', 'COMPANY', 'SECTOR', 'SUBSECTOR', 'TODAY_PRICE', 'TOMORROW_PRED_MLP', 
        'DAY5_PRED_MLP', 'EXPECTED_5D_RETURN_PCT', 'RSI_14', 
        'POLAR_MAGNITUDE', 'POLAR_ANGLE_DEG'
    ]
    
    sell_out_table = sell_out_table[target_columns].reset_index(drop=True)

# =====================================================================
# GOOGLE NEWS PIPELINE INTEGRATION (Last 48 Hours)
# =====================================================================
def get_recent_news_volume(keyword):
    if pd.isna(keyword) or not str(keyword).strip():
        return 0
        
    encoded_keyword = urllib.parse.quote(str(keyword).strip())
    rss_url = f"https://news.google.com/rss/search?q={encoded_keyword}&hl=pt-BR&gl=BR&ceid=BR:pt-419"
    
    try:
        feed = feedparser.parse(rss_url, agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64)')
        if not feed.entries:
            return 0
            
        now_utc = datetime.now(timezone.utc)
        cutoff_time = now_utc - timedelta(hours=48)
        
        recent_count = 0
        for entry in feed.entries:
            pub_date_str = entry.get('published')
            if pub_date_str:
                try:
                    pub_date = pd.to_datetime(pub_date_str, utc=True)
                    if pub_date >= cutoff_time:
                        recent_count += 1
                except Exception:
                    continue
        return recent_count
    except Exception:
        return 0

def classify_volume(count):
    if count == 0:
        return 'NO NEWS'
    elif 1 <= count <= 2:
        return 'GREEN'
    elif 3 <= count <= 6:
        return 'YELLOW'
    else:
        return 'RED'

# Execute news count loop if the pipeline successfully generated rows
if pipeline_success and not sell_out_table.empty:
    print("🔄 Pulling real-time Google News trends for sell-out targets...")
    volumes_int = []
    
    for index, row in sell_out_table.iterrows():
        raw_company_name = str(row['COMPANY']).strip()
        
        # FIXED: Wrapped with raw string r'' syntax to prevent string escape parse warnings
        clean_keyword = pd.Series([raw_company_name]).str.replace(
            r'\b(S\.?A\.?|PART\.?|INC\.?|AG\.?|CORP\.?|LTDA\.?|INT\.?|THERA\.?|THERAPEUTICS)\b', 
            '', regex=True, flags=re.IGNORECASE
        ).str.replace(r'[^\w\s-]', '', regex=True).str.strip().values[0]
        
        search_query = clean_keyword if clean_keyword else raw_company_name
        news_count = get_recent_news_volume(search_query)
        volumes_int.append(news_count)
        
    sell_out_table['VOLUME_RECENT_NEWS'] = [classify_volume(c) for c in volumes_int]

# =====================================================================
# DISPLAY RESULTS
# =====================================================================
print("=" * 140)
print(f"🚀 [FINAL STRATEGIC SELL_OUT_TABLE - {len(sell_out_table)} TOTAL ROWS SEEDED]")
print("=" * 140)
if pipeline_success and not sell_out_table.empty:
    print(sell_out_table.to_string(formatters={
        'TODAY_PRICE': 'R$ {:,.2f}'.format,
        'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
        'DAY5_PRED_MLP': 'R$ {:,.2f}'.format,
        'EXPECTED_5D_RETURN_PCT': '{:,.2f}%'.format,
        'RSI_14': '{:,.2f}'.format,
        'POLAR_MAGNITUDE': '{:,.4f}'.format,
        'POLAR_ANGLE_DEG': '{:,.2f}°'.format,
        'VOLUME_RECENT_NEWS': '{}'.format
    }))
else:
    print("⚠️ CRITICAL ALARM: Zero assets found even after applying the Stage 3 Capitulation Fallback.")
print("=" * 140)

In [ ]:
sell_out_table.head(10)

In [ ]:
#### [AI-CONTEXT]
#### ID: 037
#### ROLE: Identify the exit prices for each one of the stocks listed in the buy_in_table.
#### INPUT: Calculate the option with better buy-in prices. (successful execution of ID: 033).
#### OUTPUT: dataframe long_position_exit_price.


import pandas as pd
import numpy as np

# =====================================================================
# 1. INITIALIZE GLOBAL INFRASTRUCTURE & EXTRACT NOMINAL SCALARS
# =====================================================================
if 'buy_in_table' not in globals():
    raise NameError("❌ Sequence Error: 'buy_in_table' not found. Please run ID:034 first.")

print("🚀 Running streamlined exit router using pre-compiled ID:029 peaks...")

# Isolate the stock codes present in the buy_in_table selection
active_buy_tickers = buy_in_table['INDEX'].tolist()

# Extract parameters directly via Pandas to guarantee perfect row-by-row alignment
price_cols = [c for c in df.columns if str(c).replace('-', '').isdigit() or str(c) == '0']
historical_price_cols = [c for c in price_cols if str(c) != '0']

raw_historical_matrix = df.set_index('INDEX')[historical_price_cols].astype(float)
df_stds_dict = raw_historical_matrix.std(axis=1, ddof=0).to_dict()

autoregressive_peaks = []
selected_stds = []
selected_losses = []

# =====================================================================
# 2. PARAMETER & EXTRACTION LOOP
# =====================================================================
for idx, row in buy_in_table.iterrows():
    ticker = row['INDEX']
    
    # Locate the matching row inside your master pipeline dataframe
    matching_master_row = df[df['INDEX'] == ticker].iloc[0]
    
    # Extract the crash-proof unscaled peak ceiling pre-calculated in ID:029
    autoregressive_peaks.append(matching_master_row['PEAK_10D_PRED_MLP'])
    
    # Extract standard deviation for mathematical bands calculation
    sigma = df_stds_dict.get(ticker, 1.0) + 1e-6
    selected_stds.append(sigma)
    selected_losses.append(sigma * 0.15)

selected_stds = np.array(selected_stds)
selected_losses = np.array(selected_losses)

# =====================================================================
# 3. TECHNIQUE 2 & 3: GEOMETRIC BANDS AND RSI REVERSION TARGETS
# =====================================================================
angles_deg = buy_in_table['POLAR_ANGLE_DEG'].values
angles_rad = np.radians(angles_deg)

# Generate upper statistical and reversion price volatility barriers
geometric_exits = buy_in_table['TODAY_PRICE'].values + (selected_stds * np.sin(angles_rad) * 1.5)

target_rs_constant = 65.0 / (100.0 - 65.0)
rsi_reversion_exits = buy_in_table['TODAY_PRICE'].values + (target_rs_constant * selected_losses)

# =====================================================================
# 4. ASSEMBLE FINAL LONG POSITION EXIT PRICE DATAFRAME
# =====================================================================
long_position_exit_price = pd.DataFrame({
    'INDEX': buy_in_table['INDEX'],
    'COMPANY': buy_in_table['COMPANY'],
    'SECTOR': buy_in_table['SECTOR'],
    'SUBSECTOR': buy_in_table['SUBSECTOR'],
    'TODAY_PRICE': buy_in_table['TODAY_PRICE'],
    'TOMORROW_PRED_MLP': buy_in_table['TOMORROW_PRED_MLP'],
    'EXIT_AUTOREGRESIVE_PEAK': autoregressive_peaks,
    'EXIT_GEOMETRIC_BAND': geometric_exits,
    'EXIT_RSI_REVERSION': rsi_reversion_exits
})

# =====================================================================
# DISPLAY RESULTS
# =====================================================================
print("=" * 140)
print(f"🎯 [STRATEGIC LONG TARGETS LOCKED - {len(long_position_exit_price)} ASSETS MOUNTED CLASH-FREE]")
print("=" * 140)
print(long_position_exit_price.to_string(index=False, formatters={
    'TODAY_PRICE': 'R$ {:,.2f}'.format,
    'TOMORROW_PRED_MLP': 'R$ {:,.2f}'.format,
    'EXIT_AUTOREGRESIVE_PEAK': 'R$ {:,.2f}'.format,
    'EXIT_GEOMETRIC_BAND': 'R$ {:,.2f}'.format,
    'EXIT_RSI_REVERSION': 'R$ {:,.2f}'.format
}))
print("=" * 140)

In [ ]:
long_position_exit_price.head(10)